In [1]:
import pandas as pd
from pprint import pprint
from datetime import datetime
import json

In [2]:
df = pd.read_excel('../Y/excel/Y.xlsx', sheet_name=0, dtype=str)
df.columns = df.columns.str.strip().str.replace(r'[ ,.()/]+', '_', regex=True)
df["Дата_входного_контроля"] = pd.to_datetime(df["Дата_входного_контроля"])

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9328 entries, 0 to 9327
Data columns (total 17 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   No_                                     1714 non-null   object        
 1   Дата_входного_контроля                  1697 non-null   datetime64[ns]
 2   Акт_входного_контроля_материалов_Номер  1718 non-null   object        
 3   Номер_блока                             1573 non-null   object        
 4   Наименование_материалов                 9295 non-null   object        
 5   Проект_FCR_FCO                          4796 non-null   object        
 6   Тип_марка_стандарт_ТЗ_ГОСТ_и_т_д_       2843 non-null   object        
 7   Количество                              9258 non-null   object        
 8   Единицы_измерения                       9128 non-null   object        
 9   Изготовитель_Субподрядчик               3832 non-nul

,No_,Дата_входного_контроля,Акт_входного_контроля_материалов_Номер,Номер_блока,Наименование_материалов,Проект_FCR_FCO,Тип_марка_стандарт_ТЗ_ГОСТ_и_т_д_,Количество,Единицы_измерения,Изготовитель_Субподрядчик,Накладная,Заключение,Исполнитель_входной_проверки,Замечания_к_Акту_входной_проверки,Документы_на_изделия,Дополнительные_документы,Партия
0,2-1,2022-06-10,ED.A023.MII.1.0002-1,Unit 1,Laminated plywood 18,GOST 53920-2010;\nED.D.P034.1.0UKA99&&&&&&.015...,NaN,109.9,m2,Bryansk Plywood Mill,Packing list № 02 dated 24.03.2022 (1 page),Accepted,Shatilo Petr;\nSkirukha Vadzim;\nLubinski Raman,NaN,Quality certificate №N/A dated 22.08.2022 (1 p...,NaN,NaN
1,NaN,NaT,NaN,NaN,Telescopic rack Peri PEP Ergo D-400,Peri;\nED.D.P034.1.0UKA99&&&&&&.015.KE.0002.E ...,NaN,23,pcs,Peri,Packing list № 22072022-4 dated 22.03.2022 (2 ...,NaN,NaN,NaN,Certificate of Conformity dated 14.04.2022 (2 ...,NaN,NaN
2,NaN,NaT,NaN,NaN,Telescopic rack Peri PEP Ergo D-250,Peri;\nED.D.P034.1.0UKA99&&&&&&.015.KE.0002.E ...,NaN,30,pcs,Peri,Packing list № 22072022-4 dated 22.03.2022 (2 ...,NaN,NaN,NaN,Certificate of Conformity dated 14.04.2022 (2 ...,NaN,NaN
3,NaN,NaT,NaN,NaN,Universal tripod by Peri nomenclature (or simi...,Peri;\nED.D.P034.1.0UKA99&&&&&&.015.KE.0002.E ...,NaN,53,pcs,Peri,Packing list № 22072022-4 dated 22.03.2022 (2 ...,NaN,NaN,NaN,Certificate of Conformity dated 12.04.2022 (2 ...,NaN,NaN
4,NaN,NaT,NaN,NaN,Lowering head by Peri nomenclature (or similar),Peri;\nED.D.P034.1.0UKA99&&&&&&.015.KE.0002.E ...,NaN,28,pcs,Peri,Packing list № 22072022-4 dated 22.03.2022 (2 ...,NaN,NaN,NaN,Certificate of Conformity dated 12.04.2022 (2 ...,NaN,NaN


In [3]:
# Из эксель получить JSON

curr_name = ''
y = {}
old_name = ''
id_act = 1


def vnos(val):
    return None if pd.isna(val) else None if not val.strip() else val.strip()


def vnos_split(val):
    return None if pd.isna(val) else [x.strip() for x in val.split(';') if x]

def flatten(obj):
    if isinstance(obj, list):
        res = []
        for v in obj:
            sub_res = flatten(v)
            if isinstance(sub_res, list):
                res.extend(sub_res)
            else:
                res.append(sub_res)
        return res
    return obj


def clear_list(list_):
    if not isinstance(list_, list):
        return list_
    try:
        # list_ = sum(list_, [])
        list_ = flatten(list_)
    except:
        pass
    unic = set()
    res = []
    for l in list_:
        if l is not None:
            l_str = str(l).strip()
            if l_str not in unic:
                unic.add(l_str)
                res.append(l_str)

    return res if res else None

# def invoice_stack(list_dict):
#     # pprint(list_dict)
#     invoice = []
#     for invoice_1 in list_dict:
#         invoice.append(invoice_1['invoice'])
#     invoice_unic =  clear_list(invoice)
#     # pprint(invoice_unic)
#     if not invoice_unic:
#         invoice_unic = [None]
#     elif len(invoice_unic) == len(invoice):
#         return list_dict
#     res = []
#     for i, inv in enumerate(invoice_unic, start=1):
#         curr_project, curr_document = [], []     #curr_project, curr_document, curr_extra_document = [], [], []
#         for invoice_1 in list_dict:
#             if invoice_1['invoice'] == inv:
#                 curr_project.extend(invoice_1['project'] if invoice_1['project'] else [])
#                 curr_document.extend(invoice_1['document'] if invoice_1['document'] else [])
#                 # curr_extra_document.extend(invoice_1['extra_document'] if invoice_1['extra_document'] else [])


#         res.append(
#             {
#                 'id': i,
#                 'invoice': inv,
#                 'project': None if not curr_project else clear_list(curr_project),
#                 'document': None if not curr_project else clear_list(curr_document),
#                 # 'extra_document': None if not curr_project else clear_list(curr_extra_document)
#             }
#         )
#     # pprint(res)
#     return res


for row in df.itertuples():

    if not pd.isna(getattr(row, 'Акт_входного_контроля_материалов_Номер')):
        curr_name = getattr(row, 'Акт_входного_контроля_материалов_Номер')

    if curr_name not in y:

        if old_name:
            y[old_name]['material'] = material
            y[old_name]['manufacturer'] = clear_list(manufacturer)
            y[old_name]['documents'] = clear_list(documents)
            y[old_name]['invoices'] = clear_list(invoices)
            y[old_name]['projects'] = clear_list(projects)
            # y[old_name]['invoice'] = invoice_stack(invoice)
            # y[old_name]['extra_document'] = clear_list(extra_document)
            # break

        old_name = curr_name

        y[curr_name] = {}  # 1
        act = y[curr_name]

        act['id'] = id_act  # 1
        id_act += 1

        act['number_act'] = curr_name  # 1

        block_val = getattr(row, 'Номер_блока')
        act['block'] = vnos(block_val)  # 2

        commission_val = getattr(row, 'Исполнитель_входной_проверки')
        act['commission'] = None if pd.isna(commission_val) else [
            x.strip() for x in commission_val.split(';') if x]  # 3

        date_val = getattr(row, 'Дата_входного_контроля')
        act['date'] = None if pd.isna(
            date_val) else date_val.strftime("%Y-%m-%d")  # 4

        remarks_val = getattr(row, 'Замечания_к_Акту_входной_проверки')
        act['remarks'] = vnos(remarks_val)  # 5

        conclusion_val = getattr(row, 'Заключение')
        act['conclusion'] = vnos(conclusion_val)  # 6

        id_material = 1
        material = []
        manufacturer = []
        id_invoice = 1
        invoice = []
        documents = []
        invoices = []
        projects = []
        # extra_document = []

        # curr_invoice = getattr(row, 'Накладная')

    name_val = getattr(row, 'Наименование_материалов')
    # type_val = getattr(row, 'Тип_марка_стандарт_ТЗ_ГОСТ_и_т_д_')
    quantity_val_1 = getattr(row, 'Количество')
    quantity_val_2 = getattr(row, 'Единицы_измерения')
    batch_val = getattr(row, 'Партия')

    material_1 = {
        'id': None,
        'name': None,
        # 'type': None,
        'batch': None,
        'quantity': {
            'value': None,
            'unit': None
        }}

    material_1['id'] = id_material
    material_1['name'] = vnos(name_val)  # 7
    # material_1['type'] = vnos(type_val)  # 8
    material_1['batch'] = vnos(batch_val)  # 9
    material_1['quantity']['value'] = vnos(quantity_val_1)  # 10
    material_1['quantity']['unit'] = vnos(quantity_val_2)  # 11

    id_material += 1
    material.append(material_1)

    manufacturer_val = getattr(row, 'Изготовитель_Субподрядчик')  # 12
    manufacturer.append(vnos(manufacturer_val))

    extra_document_val = getattr(row, 'Дополнительные_документы')  # 16
    document_val = getattr(row, 'Документы_на_изделия')  # 15
    documents.append(vnos_split(extra_document_val))
    documents.append(vnos_split(document_val))
    # extra_document.append(vnos(extra_document_val))

    invoice_val = getattr(row, 'Накладная')  # 13
    invoices.append(vnos_split(invoice_val))

    project_val = getattr(row, 'Проект_FCR_FCO')  # 14
    type_val = getattr(row, 'Тип_марка_стандарт_ТЗ_ГОСТ_и_т_д_')   # 8
    projects.append(vnos_split(project_val))
    projects.append(vnos_split(type_val))

    # if  pd.isna(invoice_val):
    #     invoice_val = curr_invoice
    # else:
    #     curr_invoice = invoice_val
    # project_val = getattr(row, 'Проект_FCR_FCO')
    # document_val = getattr(row, 'Документы_на_изделия')
    # extra_document_val = getattr(row, 'Дополнительные_документы')

    # invoice_1 = {
    #     'id': None,
    #     # 'invoice': None,
    #     'project': None,
    #     # 'document': None,
    #     # 'extra_document': None
    # }

    # invoice_1['id'] = id_invoice
    # # invoice_1['invoice'] =  None if pd.isna(invoice_val) else [x.strip() for x in invoice_val.split(';') if x] #13
    # invoice_1['project'] =  None if pd.isna(project_val) else [x.strip() for x in project_val.split(';') if x]   #14
    # # invoice_1['document'] =  None if pd.isna(document_val) else [x.strip() for x in document_val.split(';') if x]   #15
    # # invoice_1['extra_document'] =  None if pd.isna(extra_document_val) else [x.strip() for x in extra_document_val.split(';') if x]   #16

    # id_invoice += 1
    # invoice.append(invoice_1)

else:
    y[old_name]['material'] = material
    y[old_name]['manufacturer'] = clear_list(manufacturer)
    y[old_name]['documents'] = clear_list(documents)
    y[old_name]['invoices'] = clear_list(invoices)
    y[old_name]['projects'] = clear_list(projects)
    # y[old_name]['invoice'] = invoice_stack(invoice)
    # y[old_name]['extra_document'] = clear_list(extra_document)

y = dict(sorted(y.items(), key=lambda x: x[1]['id']))
# pprint(material)
# pprint(y['ED.A023.MII.&.0009'])
pprint(y[list(y)[-50]])
# pprint(y)

{'block': 'Unit 0',
 'commission': ['Moiseenko Aleksandr',
                'Shkrabinskiy Aleksandr',
                'Akimova Kseniya'],
 'conclusion': 'Accepted',
 'date': '2026-09-23',
 'documents': ['Certificate of analysis/Сертификат анализа № QCA 422126 dated '
               '16.09.2026 (1 page)',
               'Certificate of analysis/Сертификат анализа № QCA 422226 dated '
               '16.09.2026 (1 page)'],
 'id': 1669,
 'invoices': ['Delivery receipt/Товарно-транспортная накладная № 352-26 dated '
              '21.09.2026 (1 page)'],
 'manufacturer': ['Modern Waterproofing Company'],
 'material': [{'batch': '0511409262',
               'id': 1,
               'name': 'Bitumode ALPHA-P(WT) roll-on waterproofing / Рулонная '
                       'наплавляемая гидроизоляция  BITUMODE ALPHA-P(WT)',
               'quantity': {'unit': 'm2/м2', 'value': '190'}},
              {'batch': '1131109263',
               'id': 2,
               'name': 'Bitumode DELTA-G roll-on wat

In [5]:
with open(r"../Y/data.json", "w", encoding="utf-8") as file:
    json.dump(y, file, ensure_ascii=False, indent=4)

In [ ]:
# Версия 2
# Из эксель получить JSON

curr_name = ''
y = {}
old_name = ''
id_act = 1


def vnos(val):
    return None if pd.isna(val) else None if not val.strip() else val.strip()


def clear_list(list_):
    if not isinstance(list_, list):
        return list_
    unic = set()
    res = []
    for l in list_:
        if l is not None:
            l_str = str(l).strip()
            if l_str not in unic:
                unic.add(l_str)
                res.append(l_str)

    return res if res else None


def invoice_stack(list_dict):
    # pprint(list_dict)
    invoice = []
    for invoice_1 in list_dict:
        invoice.append(invoice_1['invoice'])
    invoice_unic = clear_list(invoice)
    # pprint(invoice_unic)
    if not invoice_unic:
        invoice_unic = [None]
    elif len(invoice_unic) == len(invoice):
        return list_dict
    res = []
    for i, inv in enumerate(invoice_unic, start=1):
        # curr_project, curr_document, curr_extra_document = [], [], []
        curr_project, curr_document = [], []
        for invoice_1 in list_dict:
            if invoice_1['invoice'] == inv:
                curr_project.extend(
                    invoice_1['project'] if invoice_1['project'] else [])
                curr_document.extend(
                    invoice_1['document'] if invoice_1['document'] else [])
                # curr_extra_document.extend(invoice_1['extra_document'] if invoice_1['extra_document'] else [])

        res.append(
            {
                'id': i,
                'invoice': inv,
                'project': None if not curr_project else clear_list(curr_project),
                'document': None if not curr_project else clear_list(curr_document),
                # 'extra_document': None if not curr_project else clear_list(curr_extra_document)
            }
        )
    # pprint(res)
    return res


for row in df.itertuples():

    if not pd.isna(getattr(row, 'Акт_входного_контроля_материалов_Номер')):
        curr_name = getattr(row, 'Акт_входного_контроля_материалов_Номер')

    if curr_name not in y:

        if old_name:
            y[old_name]['material'] = material
            y[old_name]['manufacturer'] = clear_list(manufacturer)
            y[old_name]['invoice'] = invoice_stack(invoice)
            y[old_name]['extra_document'] = clear_list(extra_document)
            # break

        old_name = curr_name

        y[curr_name] = {}  # 1
        act = y[curr_name]

        act['id'] = id_act  # 1
        id_act += 1

        act['number_act'] = curr_name  # 1

        block_val = getattr(row, 'Номер_блока')
        act['block'] = vnos(block_val)  # 2

        commission_val = getattr(row, 'Исполнитель_входной_проверки')
        act['commission'] = None if pd.isna(commission_val) else [
            x.strip() for x in commission_val.split(';') if x]  # 3

        date_val = getattr(row, 'Дата_входного_контроля')
        act['date'] = None if pd.isna(
            date_val) else date_val.strftime("%Y-%m-%d")  # 4

        remarks_val = getattr(row, 'Замечания_к_Акту_входной_проверки')
        act['remarks'] = vnos(remarks_val)  # 5

        conclusion_val = getattr(row, 'Заключение')
        act['conclusion'] = vnos(conclusion_val)  # 6

        id_material = 1
        material = []
        manufacturer = []
        id_invoice = 1
        invoice = []
        extra_document = []

        curr_invoice = getattr(row, 'Накладная')

    name_val = getattr(row, 'Наименование_материалов')
    type_val = getattr(row, 'Тип_марка_стандарт_ТЗ_ГОСТ_и_т_д_')
    quantity_val_1 = getattr(row, 'Количество')
    quantity_val_2 = getattr(row, 'Единицы_измерения')
    batch_val = getattr(row, 'Партия')

    material_1 = {
        'id': None,
        'name': None,
        'type': None,
        'batch': None,
        'quantity': {
            'value': None,
            'unit': None
        }}

    material_1['id'] = id_material
    material_1['name'] = vnos(name_val)  # 7
    material_1['type'] = vnos(type_val)  # 8
    material_1['batch'] = vnos(batch_val)  # 9
    material_1['quantity']['value'] = vnos(quantity_val_1)  # 10
    material_1['quantity']['unit'] = vnos(quantity_val_2)  # 11

    id_material += 1
    material.append(material_1)

    manufacturer_val = getattr(row, 'Изготовитель_Субподрядчик')  # 12
    manufacturer.append(vnos(manufacturer_val))

    extra_document_val = getattr(row, 'Дополнительные_документы')
    extra_document.append(vnos(extra_document_val))

    invoice_val = getattr(row, 'Накладная')
    if pd.isna(invoice_val):
        invoice_val = curr_invoice
    else:
        curr_invoice = invoice_val
    project_val = getattr(row, 'Проект_FCR_FCO')
    document_val = getattr(row, 'Документы_на_изделия')
    # extra_document_val = getattr(row, 'Дополнительные_документы')

    invoice_1 = {
        'id': None,
        'invoice': None,
        'project': None,
        'document': None,
        # 'extra_document': None
    }

    invoice_1['id'] = id_invoice
    invoice_1['invoice'] = None if pd.isna(invoice_val) else [
        x.strip() for x in invoice_val.split(';') if x]  # 13
    invoice_1['project'] = None if pd.isna(project_val) else [
        x.strip() for x in project_val.split(';') if x]  # 14
    invoice_1['document'] = None if pd.isna(document_val) else [
        x.strip() for x in document_val.split(';') if x]  # 15
    # invoice_1['extra_document'] =  None if pd.isna(extra_document_val) else [x.strip() for x in extra_document_val.split(';') if x]   #16

    id_invoice += 1
    invoice.append(invoice_1)

else:
    y[old_name]['material'] = material
    y[old_name]['manufacturer'] = clear_list(manufacturer)
    y[old_name]['invoice'] = invoice_stack(invoice)
    y[old_name]['extra_document'] = clear_list(extra_document)

y = dict(sorted(y.items(), key=lambda x: x[1]['id']))
# pprint(material)
# pprint(y['ED.A009.MII.&.0015'])
pprint(y[list(y)[-1]])
# pprint(y)

In [ ]:
# Версия 1
# Из эксель получить JSON

curr_name = ''
y = {}
old_name = ''
id_act = 1


def vnos(val):
    return None if pd.isna(val) else None if not val.strip() else val.strip()


def clear_list(list_):
    if not isinstance(list_, list):
        return list_
    unic = set()
    res = []
    for l in list_:
        if l is not None:
            l_str = str(l).strip()
            if l_str not in unic:
                unic.add(l_str)
                res.append(l_str)

    return res if res else None


def invoice_stack(list_dict):
    # pprint(list_dict)
    invoice = []
    for invoice_1 in list_dict:
        invoice.append(invoice_1['invoice'])
    invoice_unic = clear_list(invoice)
    # pprint(invoice_unic)
    if not invoice_unic:
        invoice_unic = [None]
    elif len(invoice_unic) == len(invoice):
        return list_dict
    res = []
    for i, inv in enumerate(invoice_unic, start=1):
        curr_project, curr_document, curr_extra_document = [], [], []
        for invoice_1 in list_dict:
            if invoice_1['invoice'] == inv:
                curr_project.extend(
                    invoice_1['project'] if invoice_1['project'] else [])
                curr_document.extend(
                    invoice_1['document'] if invoice_1['document'] else [])
                curr_extra_document.extend(
                    invoice_1['extra_document'] if invoice_1['extra_document'] else [])

        res.append(
            {
                'id': i,
                'invoice': inv,
                'project': None if not curr_project else clear_list(curr_project),
                'document': None if not curr_project else clear_list(curr_document),
                'extra_document': None if not curr_project else clear_list(curr_extra_document)
            }
        )
    # pprint(res)
    return res


for row in df.itertuples():

    if not pd.isna(getattr(row, 'Акт_входного_контроля_материалов_Номер')):
        curr_name = getattr(row, 'Акт_входного_контроля_материалов_Номер')

    if curr_name not in y:

        if old_name:
            y[old_name]['material'] = material
            y[old_name]['manufacturer'] = clear_list(manufacturer)
            y[old_name]['invoice'] = invoice_stack(invoice)
            # break

        old_name = curr_name

        y[curr_name] = {}  # 1
        act = y[curr_name]

        act['id'] = id_act  # 1
        id_act += 1

        act['number_act'] = curr_name  # 1

        block_val = getattr(row, 'Номер_блока')
        act['block'] = vnos(block_val)  # 2

        commission_val = getattr(row, 'Исполнитель_входной_проверки')
        act['commission'] = None if pd.isna(commission_val) else [
            x.strip() for x in commission_val.split(';') if x]  # 3

        date_val = getattr(row, 'Дата_входного_контроля')
        act['date'] = None if pd.isna(
            date_val) else date_val.strftime("%Y-%m-%d")  # 4

        remarks_val = getattr(row, 'Замечания_к_Акту_входной_проверки')
        act['remarks'] = vnos(remarks_val)  # 5

        conclusion_val = getattr(row, 'Заключение')
        act['conclusion'] = vnos(conclusion_val)  # 6

        id_material = 1
        material = []
        manufacturer = []
        id_invoice = 1
        invoice = []

        curr_invoice = getattr(row, 'Накладная')

    name_val = getattr(row, 'Наименование_материалов')
    type_val = getattr(row, 'Тип_марка_стандарт_ТЗ_ГОСТ_и_т_д_')
    quantity_val_1 = getattr(row, 'Количество')
    quantity_val_2 = getattr(row, 'Единицы_измерения')
    batch_val = getattr(row, 'Партия')

    material_1 = {
        'id': None,
        'name': None,
        'type': None,
        'batch': None,
        'quantity': {
            'value': None,
            'unit': None
        }}

    material_1['id'] = id_material
    material_1['name'] = vnos(name_val)  # 7
    material_1['type'] = vnos(type_val)  # 8
    material_1['batch'] = vnos(batch_val)  # 9
    material_1['quantity']['value'] = vnos(quantity_val_1)  # 10
    material_1['quantity']['unit'] = vnos(quantity_val_2)  # 11

    id_material += 1
    material.append(material_1)

    manufacturer_val = getattr(row, 'Изготовитель_Субподрядчик')  # 12
    manufacturer.append(vnos(manufacturer_val))

    invoice_val = getattr(row, 'Накладная')
    if pd.isna(invoice_val):
        invoice_val = curr_invoice
    else:
        curr_invoice = invoice_val
    project_val = getattr(row, 'Проект_FCR_FCO')
    document_val = getattr(row, 'Документы_на_изделия')
    extra_document_val = getattr(row, 'Дополнительные_документы')

    invoice_1 = {
        'id': None,
        'invoice': None,
        'project': None,
        'document': None,
        'extra_document': None
    }

    invoice_1['id'] = id_invoice
    invoice_1['invoice'] = None if pd.isna(invoice_val) else [
        x.strip() for x in invoice_val.split(';') if x]  # 13
    invoice_1['project'] = None if pd.isna(project_val) else [
        x.strip() for x in project_val.split(';') if x]  # 14
    invoice_1['document'] = None if pd.isna(document_val) else [
        x.strip() for x in document_val.split(';') if x]  # 15
    invoice_1['extra_document'] = None if pd.isna(extra_document_val) else [
        x.strip() for x in extra_document_val.split(';') if x]  # 16

    id_invoice += 1
    invoice.append(invoice_1)

else:
    y[old_name]['material'] = material
    y[old_name]['manufacturer'] = clear_list(manufacturer)
    y[old_name]['invoice'] = invoice_stack(invoice)

y = dict(sorted(y.items(), key=lambda x: x[1]['id']))
# pprint(material)
# pprint(y['ED.A009.MII.&.0015'])
# pprint(y[list(y)[-55]])
# pprint(y)

In [18]:
import os

# 1. Указываем путь к папке (буква 'r' перед строкой отменяет экранирование слешей)
folder_path = r"../X"

# 2. Получаем список ВШЕХ объектов в этой папке (и файлов, и других папок)
all_items = os.listdir(folder_path)

# 3. Фильтруем: оставляем только файлы
only_files = []
for item in all_items:
    # Собираем полный путь к объекту (например, "C:\folder\file.txt")
    full_path = os.path.join(folder_path, item)

    # Проверяем, является ли этот объект файлом (а не папкой)
    if os.path.isfile(full_path):
        only_files.append(item)

# 4. Выводим результат
print(f"Всего файлов найдено: {len(only_files)}")
# print(only_files)

set_need_files = set(need_files)

need_delete = []

for f in only_files:
    if f not in set_need_files:
        need_delete.append(f)

print(len(need_delete))
print(len(only_files) - len(need_delete))
# need_delete
for file_name in need_delete:

    # Собираем полный путь к файлу для удаления
    file_to_delete = os.path.join(folder_path, file_name)

    os.remove(file_to_delete)  # Удалени

Всего файлов найдено: 1718
0
1718


In [17]:
need_files = [
    "ED.A009.MII.&,012.0566 (eng) от 04.12.2023 - в 15.48.docx",
    "ED.A009.MII.&.0015.docx",
    "ED.A009.MII.&.012.0028 (eng).docx",
    "ED.A009.MII.&.012.0057 (eng).docx",
    "ED.A009.MII.&.012.0143 (eng).docx",
    "ED.A009.MII.&.012.0156 (eng) от 26.05.2024 - в 08.43.docx",
    "ED.A009.MII.&.012.0157 (eng).docx",
    "ED.A009.MII.&.012.0172 (eng) от 12.06.2023 - в 09.19.docx",
    "ED.A009.MII.&.012.0173 (eng) от 01.06.2023 - в 14.35.docx",
    "ED.A009.MII.&.012.0177 (eng).docx",
    "ED.A009.MII.&.012.023 (eng).docx",
    "ED.A009.MII.&.012.0274.docx",
    "ED.A009.MII.&.012.0285 (eng).docx",
    "ED.A009.MII.&.012.0296 (eng) от 26.03.2025 - в 12.58.docx",
    "ED.A009.MII.&.012.0306 (eng) от 17.08.2023 - в 11.02.docx",
    "ED.A009.MII.&.012.0312 (eng) от 24.08.2023 - в 09.32.docx",
    "ED.A009.MII.&.012.0314 (eng) от 24.08.2023 - в 13.18.docx",
    "ED.A009.MII.&.012.0315.docx",
    "ED.A009.MII.&.012.0325.docx",
    "ED.A009.MII.&.012.0326 (eng).docx",
    "ED.A009.MII.&.012.0334.docx",
    "ED.A009.MII.&.012.0337 (eng) от 11.09.2023 - в 10.30.docx",
    "ED.A009.MII.&.012.0348 (eng) от 26.07.2024 - в 13.03.docx",
    "ED.A009.MII.&.012.0349 (eng).docx",
    "ED.A009.MII.&.012.0368 (eng) от 13.05.2024 - в 09.08.docx",
    "ED.A009.MII.&.012.0370 (eng) от 02.10.2023 - в 16.35.docx",
    "ED.A009.MII.&.012.0371 (eng) от 17.05.2024 - в 10.58.docx",
    "ED.A009.MII.&.012.0372 (eng).docx",
    "ED.A009.MII.&.012.0373 (eng).docx",
    "ED.A009.MII.&.012.0374 (eng) от 03.10.2023 - в 15.02.docx",
    "ED.A009.MII.&.012.0377 (eng) от 04.10.2023 - в 10.50.docx",
    "ED.A009.MII.&.012.0381 (eng) от 25.07.2024 - в 13.01.docx",
    "ED.A009.MII.&.012.0385 (eng) от 09.10.2023 - в 17.53.docx",
    "ED.A009.MII.&.012.0386 (eng).docx",
    "ED.A009.MII.&.012.0387 (eng).docx",
    "ED.A009.MII.&.012.0388 (eng).docx",
    "ED.A009.MII.&.012.0396 (eng).docx",
    "ED.A009.MII.&.012.0397 (eng).docx",
    "ED.A009.MII.&.012.0398 (eng).docx",
    "ED.A009.MII.&.012.0399 (eng).docx",
    "ED.A009.MII.&.012.0400 (eng).docx",
    "ED.A009.MII.&.012.0401 (eng) от 16.10.2023 - в 17.52.docx",
    "ED.A009.MII.&.012.0403 (eng).docx",
    "ED.A009.MII.&.012.0407.docx",
    "ED.A009.MII.&.012.0408 (eng).docx",
    "ED.A009.MII.&.012.0409 (eng).docx",
    "ED.A009.MII.&.012.0411 (eng).docx",
    "ED.A009.MII.&.012.0414 (eng) от 26.10.2023 - в 14.30.docx",
    "ED.A009.MII.&.012.0415 (eng) русск.docx",
    "ED.A009.MII.&.012.0417 (eng).docx",
    "ED.A009.MII.&.012.0418 (eng) от 26.10.2023 - в 08.22.docx",
    "ED.A009.MII.&.012.0420 (2).docx",
    "ED.A009.MII.&.012.0421.docx",
    "ED.A009.MII.&.012.0422 (eng).docx",
    "ED.A009.MII.&.012.0423 (eng).docx",
    "ED.A009.MII.&.012.0425 (eng).docx",
    "ED.A009.MII.&.012.0426.docx",
    "ED.A009.MII.&.012.0427 (eng).docx",
    "ED.A009.MII.&.012.0428.docx",
    "ED.A009.MII.&.012.0429.docx",
    "ED.A009.MII.&.012.0430 (eng).docx",
    "ED.A009.MII.&.012.0431 (eng).docx",
    "ED.A009.MII.&.012.0432 (eng) от 24.10.2023 - в 17.41.docx",
    "ED.A009.MII.&.012.0433 (eng).docx",
    "ED.A009.MII.&.012.0434 (eng).docx",
    "ED.A009.MII.&.012.0435 (eng).docx",
    "ED.A009.MII.&.012.0436 (eng) от 25.10.2023 - в 10.54.docx",
    "ED.A009.MII.&.012.0438.docx",
    "ED.A009.MII.&.012.0440.docx",
    "ED.A009.MII.&.012.0441.docx",
    "ED.A009.MII.&.012.0442 (eng) от 20.05.2024 - в 09.11.docx",
    "ED.A009.MII.&.012.0443.docx",
    "ED.A009.MII.&.012.0444 (eng) от 27.10.2023 - в 10.01.docx",
    "ED.A009.MII.&.012.0445 (eng) от 27.10.2023 - в 13.52.docx",
    "ED.A009.MII.&.012.0446 (eng) от 27.10.2023 - в 14.18.docx",
    "ED.A009.MII.&.012.0447.docx",
    "ED.A009.MII.&.012.0448 (eng).docx",
    "ED.A009.MII.&.012.0449 (eng).docx",
    "ED.A009.MII.&.012.0452.docx",
    "ED.A009.MII.&.012.0453 (eng).docx",
    "ED.A009.MII.&.012.0454.docx",
    "ED.A009.MII.&.012.0455 (eng) от 31.10.2023 - в 09.16.docx",
    "ED.A009.MII.&.012.0456 (eng).docx",
    "ED.A009.MII.&.012.0457 (eng) от 31.10.2023 - в 10.16.docx",
    "ED.A009.MII.&.012.0458.docx",
    "ED.A009.MII.&.012.0459.docx",
    "ED.A009.MII.&.012.0460 (eng) от 01.11.2023 - в 13.41.docx",
    "ED.A009.MII.&.012.0461 (eng) от 01.11.2023 - в 15.20.docx",
    "ED.A009.MII.1.030.0174 (eng) от 04.06.2023 - в 14.26.docx",
    "ED.A009.MII.&.012.0462.docx",
    "ED.A009.MII.&.012.0463.docx",
    "ED.A009.MII.&.012.0464 (eng) от 01.11.2023 - в 11.01.docx",
    "ED.A009.MII.&.012.0465 (eng).docx",
    "ED.A009.MII.&.012.0466 (eng).docx",
    "ED.A009.MII.&.012.0467.docx",
    "ED.A009.MII.&.012.0468.docx",
    "ED.A009.MII.&.012.0469 (eng) от 02.11.2023 - в 10.08.docx",
    "ED.A009.MII.&.012.0470 (eng).docx",
    "ED.A009.MII.&.012.0471.docx",
    "ED.A009.MII.&.012.0472 (eng).docx",
    "ED.A009.MII.&.012.0473.docx",
    "ED.A009.MII.&.012.0474 (eng).docx",
    "ED.A009.MII.&.012.0475 (eng).docx",
    "ED.A009.MII.&.012.0476.docx",
    "ED.A009.MII.&.012.0477.docx",
    "ED.A009.MII.&.012.0478.docx",
    "ED.A009.MII.&.012.0479.docx",
    "ED.A009.MII.&.012.0480 (eng) от 06.11.2023 - в 11.55.docx",
    "ED.A009.MII.&.012.0481 (eng) от 06.11.2023 - в 10.20.docx",
    "ED.A009.MII.&.012.0482 (eng) от 06.11.2023 - в 11.20.docx",
    "ED.A009.MII.&.012.0483 (eng) от 06.11.2023 - в 11.52.docx",
    "ED.A009.MII.&.012.0484 (eng) от 06.11.2023 - в 12.27.docx",
    "ED.A009.MII.&.012.0485 (eng) от 17.11.2023 - в 09.10.docx",
    "ED.A009.MII.&.012.0486 (eng) от 19.11.2023 - в 09.26.docx",
    "ED.A009.MII.&.012.0487 (eng) от 19.11.2023 - в 09.31.docx",
    "ED.A009.MII.&.012.0488 (eng) от 19.11.2023 - в 09.32.docx",
    "ED.A009.MII.&.012.0489.docx",
    "ED.A009.MII.&.012.0490 (eng).docx",
    "ED.A009.MII.&.012.0491 (eng) от 19.11.2023 - в 09.33.docx",
    "ED.A009.MII.&.012.0492 (eng) от 19.11.2023 - в 09.34.docx",
    "ED.A009.MII.&.012.0493 (eng) от 07.11.2023 - в 11.44.docx",
    "ED.A009.MII.&.012.0494 (eng) от 19.11.2023 - в 09.35.docx",
    "ED.A009.MII.&.012.0495 (eng) от 07.11.2023 - в 13.07.docx",
    "ED.A009.MII.&.012.0496 (eng).docx",
    "ED.A009.MII.&.012.0497 (eng).docx",
    "ED.A009.MII.&.012.0499 (eng) от 19.11.2023 - в 09.36.docx",
    "ED.A009.MII.&.012.0500 (eng) от 19.11.2023 - в 09.37.docx",
    "ED.A009.MII.&.012.0501 (eng) от 19.11.2023 - в 09.38.docx",
    "ED.A009.MII.&.012.0502 (eng).docx",
    "ED.A009.MII.&.012.0503.docx",
    "ED.A009.MII.&.012.0504 отриц.docx",
    "ED.A009.MII.&.012.0505 (eng) от 19.11.2023 - в 09.39.docx",
    "ED.A009.MII.&.012.0506.docx",
    "ED.A009.MII.&.012.0507 (eng).docx",
    "ED.A009.MII.&.012.0508 (eng).docx",
    "ED.A009.MII.&.012.0509 (eng).docx",
    "ED.A009.MII.&.012.0510 (eng).docx",
    "ED.A009.MII.&.012.0511 (eng).docx",
    "ED.A009.MII.&.012.0512.docx",
    "ED.A009.MII.&.012.0513.docx",
    "ED.A009.MII.&.012.0514.docx",
    "ED.A009.MII.&.012.0515 (eng).docx",
    "ED.A009.MII.&.012.0516 (eng).docx",
    "ED.A009.MII.&.012.0517 (eng).docx",
    "ED.A009.MII.&.012.0518 (eng).docx",
    "ED.A009.MII.&.012.0519 (eng).docx",
    "ED.A009.MII.&.012.0520 (eng) от 13.11.2023 - в 10.57.docx",
    "ED.A009.MII.&.012.0521 (eng).docx",
    "ED.A009.MII.&.012.0522.docx",
    "ED.A009.MII.&.012.0523 (eng).docx",
    "ED.A009.MII.&.012.0524.docx",
    "ED.A009.MII.&.012.0525.docx",
    "ED.A009.MII.&.012.0526.docx",
    "ED.A009.MII.&.012.0527.docx",
    "ED.A009.MII.&.012.0528 (eng).docx",
    "ED.A009.MII.&.012.0529 (eng).docx",
    "ED.A009.MII.&.012.0530 (eng).docx",
    "Ed.A009.MII.&.012.0532 (eng) от 26.11.2023 - в 10.25.docx",
    "ED.A009.MII.&.012.0533 (eng).docx",
    "ED.A009.MII.&.012.0534.docx",
    "ED.A009.MII.&.012.0535 (eng).docx",
    "ED.A009.MII.&.012.0536.docx",
    "ED.A009.MII.&.012.0537.docx",
    "ED.A009.MII.&.012.0538.docx",
    "ED.A009.MII.&.012.0539.docx",
    "ED.A009.MII.&.012.0540.docx",
    "ED.A009.MII.&.012.0541 (eng).docx",
    "ED.A009.MII.&.012.0542.docx",
    "ED.A009.MII.&.012.0543 (eng) от 26.11.2023 - в 10.50.docx",
    "ED.A009.MII.&.012.0544.docx",
    "ED.A009.MII.&.012.0545.docx",
    "ED.A009.MII.&.012.0546.docx",
    "ED.A009.MII.&.012.0547 (eng).docx",
    "ED.A009.MII.&.012.0548.docx",
    "ED.A009.MII.&.012.0549.docx",
    "ED.A009.MII.&.012.0551.docx",
    "ED.A009.MII.&.012.0552 старый.docx",
    "ED.A009.MII.&.012.0553.docx",
    "ED.A009.MII.&.012.0554.docx",
    "ED.A009.MII.&.012.0555 (eng).docx",
    "ED.A009.MII.&.012.0556 старый.docx",
    "ED.A009.MII.&.012.0557.docx",
    "ED.A009.MII.&.012.0558 (eng).docx",
    "ED.A009.MII.&.012.0559.docx",
    "ED.A009.MII.&.012.0560.docx",
    "ED.A009.MII.&.012.0561 (eng).docx",
    "ED.A009.MII.&.012.0562 (eng).docx",
    "ED.A009.MII.&.012.0563.docx",
    "ED.A009.MII.&.012.0565.docx",
    "ED.A009.MII.&.012.0567 (eng).docx",
    "ED.A009.MII.&.012.0568 (eng).docx",
    "ED.A009.MII.&.012.0570 (eng).docx",
    "ED.A009.MII.&.012.0571 (eng).docx",
    "ED.A009.MII.&.012.0572.docx",
    "ED.A009.MII.&.012.0575.docx",
    "ED.A009.MII.&.012.0576.docx",
    "ED.A009.MII.&.012.0577.docx",
    "ED.A009.MII.&.012.0578 (eng).docx",
    "ED.A009.MII.&.012.0579.docx",
    "ED.A009.MII.&.012.0580 (eng).docx",
    "ED.A009.MII.&.012.0581 (eng).docx",
    "ED.A009.MII.&.012.0583.docx",
    "ED.A009.MII.&.012.0584.docx",
    "ED.A009.MII.&.012.0586.docx",
    "ED.A009.MII.&.012.0587 (eng).docx",
    "ED.A009.MII.&.012.0588.docx",
    "ED.A009.MII.&.012.0589.docx",
    "ED.A009.MII.&.012.0590.docx",
    "ED.A009.MII.&.012.0591.docx",
    "ED.A009.MII.&.012.0592.docx",
    "ED.A009.MII.&.012.0593.docx",
    "ED.A009.MII.&.012.0594 (испр).docx",
    "ED.A009.MII.&.012.0595.docx",
    "ED.A009.MII.&.012.0597.docx",
    "ED.A009.MII.&.012.0598 (eng).docx",
    "ED.A009.MII.&.012.0599.docx",
    "ED.A009.MII.&.012.0600.docx",
    "ED.A009.MII.&.012.0603 (eng) от 27.12.2023 - в 10.04.docx",
    "ED.A009.MII.&.012.0604.docx",
    "ED.A009.MII.&.012.0605 (eng).docx",
    "ED.A009.MII.&.012.0606.docx",
    "ED.A009.MII.&.012.0607 (eng).docx",
    "ED.A009.MII.&.012.0608.docx",
    "ED.A009.MII.&.012.0609 (eng) от 02.01.2024 - в 16.18.docx",
    "ED.A009.MII.&.012.0610.docx",
    "ED.A009.MII.&.012.0611.docx",
    "ED.A009.MII.&.012.0612 (eng).docx",
    "ED.A009.MII.&.012.0613 (eng).docx",
    "ED.A009.MII.&.012.0614 (eng) от 04.01.2024 - в 09.40.docx",
    "ED.A009.MII.&.012.0616.docx",
    "ED.A009.MII.&.012.0617.docx",
    "ED.A009.MII.&.012.0618.docx",
    "ED.A009.MII.&.012.0619 (eng).docx",
    "ED.A009.MII.&.012.0620.docx",
    "ED.A009.MII.&.012.0621.docx",
    "ED.A009.MII.&.012.0622 (eng) от 12.01.2024 - в 13.00.docx",
    "ED.A009.MII.&.012.0623 (eng).docx",
    "ED.A009.MII.&.012.0625 (eng).docx",
    "ED.A009.MII.&.012.0626 (eng).docx",
    "ED.A009.MII.&.012.0627.docx",
    "ED.A009.MII.&.012.0628.docx",
    "ED.A009.MII.&.012.0629 (eng).docx",
    "ED.A009.MII.&.012.0630 (eng).docx",
    "ED.A009.MII.&.012.0631 (eng).docx",
    "ED.A009.MII.&.012.0632.docx",
    "ED.A009.MII.&.012.0633.docx",
    "ED.A009.MII.&.012.0634 (eng).docx",
    "ED.A009.MII.&.012.0635 (eng).docx",
    "ED.A009.MII.&.012.0636.docx",
    "ED.A009.MII.&.012.0637.docx",
    "ED.A009.MII.&.012.0638.docx",
    "ED.A009.MII.&.012.0639.docx",
    "ED.A009.MII.&.012.0641 (eng).docx",
    "ED.A009.MII.&.012.0642.docx",
    "ED.A009.MII.&.012.0643.docx",
    "ED.A009.MII.&.012.0644.docx",
    "ED.A009.MII.&.012.0645.docx",
    "ED.A009.MII.&.012.0646 (eng).docx",
    "ED.A009.MII.&.012.0647.docx",
    "ED.A009.MII.&.012.0648.docx",
    "ED.A009.MII.&.012.0649.docx",
    "ED.A009.MII.&.012.0650.docx",
    "ED.A009.MII.&.012.0651.docx",
    "ED.A009.MII.&.012.0652 (eng).docx",
    "ED.A009.MII.&.012.0653.docx",
    "ED.A009.MII.&.012.0654 (eng) от 23.01.2024 - в 09.52.docx",
    "ED.A009.MII.&.012.0655.docx",
    "ED.A009.MII.&.012.0657.docx",
    "ED.A009.MII.&.012.0658.docx",
    "ED.A009.MII.&.012.0659.docx",
    "ED.A009.MII.&.012.0660.docx",
    "ED.A009.MII.&.012.0661.docx",
    "ED.A009.MII.&.012.0662.docx",
    "ED.A009.MII.&.012.0663 (eng).docx",
    "ED.A009.MII.&.012.0665 (eng).docx",
    "ED.A009.MII.&.012.0667 (eng).docx",
    "ED.A009.MII.&.012.0669.docx",
    "ED.A009.MII.&.012.0670.docx",
    "ED.A009.MII.&.012.0671.docx",
    "ED.A009.MII.&.012.0672 (eng).docx",
    "ED.A009.MII.&.012.0673 (eng).docx",
    "ED.A009.MII.&.012.0674.docx",
    "ED.A009.MII.&.012.0675.docx",
    "ED.A009.MII.&.012.0677 (eng).docx",
    "ED.A009.MII.&.012.0678.docx",
    "ED.A009.MII.&.012.0679.docx",
    "ED.A009.MII.&.012.0680 (eng).docx",
    "ED.A009.MII.&.012.0682.docx",
    "ED.A009.MII.&.012.0683.docx",
    "ED.A009.MII.&.012.0684 (eng).docx",
    "ED.A009.MII.&.012.0685.docx",
    "ED.A009.MII.&.012.0686.docx",
    "ED.A009.MII.&.012.0689.docx",
    "ED.A009.MII.&.012.0690.docx",
    "ED.A009.MII.&.012.0692.docx",
    "ED.A009.MII.&.012.0693 (eng).docx",
    "ED.A009.MII.&.012.0694 (eng).docx",
    "ED.A009.MII.&.012.0695 (eng).docx",
    "ED.A009.MII.&.012.0696.docx",
    "ED.A009.MII.&.012.0697.docx",
    "ED.A009.MII.&.012.0698.docx",
    "ED.A009.MII.&.012.0699.docx",
    "ED.A009.MII.&.012.0701.docx",
    "ED.A009.MII.&.012.0702.docx",
    "ED.A009.MII.&.012.0703.docx",
    "ED.A009.MII.&.012.0704.docx",
    "ED.A009.MII.&.012.0705.docx",
    "ED.A009.MII.&.012.0706.docx",
    "ED.A009.MII.&.012.0707.docx",
    "ED.A009.MII.&.012.0708.docx",
    "ED.A009.MII.&.012.0709.docx",
    "ED.A009.MII.&.012.0710.docx",
    "ED.A009.MII.&.012.0711.docx",
    "ED.A009.MII.&.012.0712.docx",
    "ED.A009.MII.&.012.0713.docx",
    "ED.A009.MII.&.012.0714.docx",
    "ED.A009.MII.&.012.0715.docx",
    "ED.A009.MII.&.012.0717.docx",
    "ED.A009.MII.&.012.0718.docx",
    "ED.A009.MII.&.012.0720.docx",
    "ED.A009.MII.&.012.0721 (eng).docx",
    "ED.A009.MII.&.012.0723.docx",
    "ED.A009.MII.&.012.0724.docx",
    "ED.A009.MII.&.012.0725.docx",
    "ED.A009.MII.&.012.0726.docx",
    "ED.A009.MII.&.012.0727.docx",
    "ED.A009.MII.&.012.0728.docx",
    "ED.A009.MII.&.012.0729.docx",
    "ED.A009.MII.&.012.0730.docx",
    "ED.A009.MII.&.012.0731.docx",
    "ED.A009.MII.&.012.0732.docx",
    "ED.A009.MII.&.012.0733.docx",
    "ED.A009.MII.&.012.0734.docx",
    "ED.A009.MII.&.012.0735.docx",
    "ED.A009.MII.&.012.0736.docx",
    "ED.A009.MII.&.012.0737.docx",
    "ED.A009.MII.&.012.0738.docx",
    "ED.A009.MII.&.012.0739.docx",
    "ED.A009.MII.&.012.0740.docx",
    "ED.A009.MII.&.012.0741.docx",
    "ED.A009.MII.&.012.0742.docx",
    "ED.A009.MII.&.012.0743 (eng).docx",
    "ED.A009.MII.&.012.0744.docx",
    "ED.A009.MII.&.012.0745.docx",
    "ED.A009.MII.&.012.0746.docx",
    "ED.A009.MII.&.012.0747.docx",
    "ED.A009.MII.&.012.0748 (eng).docx",
    "ED.A009.MII.&.012.0749.docx",
    "ED.A009.MII.&.012.0750.docx",
    "ED.A009.MII.&.012.0751 (eng).docx",
    "ED.A009.MII.&.012.0752.docx",
    "ED.A009.MII.&.012.0753.docx",
    "ED.A009.MII.&.012.0754 (eng).docx",
    "ED.A009.MII.&.012.0755.docx",
    "ED.A009.MII.&.012.0756 (eng).docx",
    "ED.A009.MII.&.012.0757 (eng).docx",
    "ED.A009.MII.&.012.0758 (eng).docx",
    "ED.A009.MII.&.012.0759 (eng).docx",
    "ED.A009.MII.&.012.0760 (eng).docx",
    "ED.A009.MII.&.012.0761.docx",
    "ED.A009.MII.&.012.0762.docx",
    "ED.A009.MII.&.012.0763.docx",
    "ED.A009.MII.&.012.0764.docx",
    "ED.A009.MII.&.012.0765.docx",
    "ED.A009.MII.&.012.0766.docx",
    "ED.A009.MII.&.012.0767.docx",
    "ED.A009.MII.&.012.0768.docx",
    "ED.A009.MII.&.012.0769.docx",
    "ED.A009.MII.&.012.0770.docx",
    "ED.A009.MII.&.012.0771.docx",
    "ED.A009.MII.&.012.0772.docx",
    "ED.A009.MII.&.012.0773 (eng).docx",
    "ED.A009.MII.&.012.0774.docx",
    "ED.A009.MII.&.012.0775.docx",
    "ED.A009.MII.&.012.0776.docx",
    "ED.A009.MII.&.012.0777.docx",
    "ED.A009.MII.&.012.0778.docx",
    "ED.A009.MII.&.012.0779.docx",
    "ED.A009.MII.&.012.0780.docx",
    "ED.A009.MII.&.012.0781.docx",
    "ED.A009.MII.&.012.0782 (eng).docx",
    "ED.A009.MII.&.012.0783.docx",
    "ED.A009.MII.&.012.0784.docx",
    "ED.A009.MII.&.012.0785.docx",
    "ED.A009.MII.&.012.0786.docx",
    "ED.A009.MII.&.012.0787.docx",
    "ED.A009.MII.&.012.0788.docx",
    "ED.A009.MII.&.012.0789.docx",
    "ED.A009.MII.&.012.0790 (eng).docx",
    "ED.A009.MII.&.012.0791.docx",
    "ED.A009.MII.&.012.0792 (eng).docx",
    "ED.A009.MII.&.012.0793.docx",
    "ED.A009.MII.&.012.0794.docx",
    "ED.A009.MII.&.012.0795.docx",
    "ED.A009.MII.&.012.0796.docx",
    "ED.A009.MII.&.012.0797.docx",
    "ED.A009.MII.&.012.0799 (eng).docx",
    "ED.A009.MII.&.012.0801.docx",
    "ED.A009.MII.&.012.0802.docx",
    "ED.A009.MII.&.012.0803-1.docx",
    "ED.A009.MII.&.012.0803.docx",
    "ED.A009.MII.&.012.0804 (eng) от 29.03.2024 - в 09.00.docx",
    "ED.A009.MII.&.012.0805.docx",
    "ED.A009.MII.&.012.0806.docx",
    "ED.A009.MII.&.012.0807.docx",
    "ED.A009.MII.&.012.0808 (eng).docx",
    "ED.A009.MII.&.012.0809.docx",
    "ED.A009.MII.&.012.0810.docx",
    "ED.A009.MII.&.012.0811.docx",
    "ED.A009.MII.&.012.0812.docx",
    "ED.A009.MII.&.012.0813.docx",
    "ED.A009.MII.&.012.0814.docx",
    "ED.A009.MII.&.012.0815.docx",
    "ED.A009.MII.&.012.0816.docx",
    "ED.A009.MII.&.012.0817 (eng).docx",
    "ED.A009.MII.&.012.0818.docx",
    "ED.A009.MII.&.012.0819.docx",
    "ED.A009.MII.&.012.0820R (eng).docx",
    "ED.A009.MII.&.012.0822.docx",
    "ED.A009.MII.&.012.0823.docx",
    "ED.A009.MII.&.012.0824.docx",
    "ED.A009.MII.&.012.0825 (eng).docx",
    "ED.A009.MII.&.012.0826 (eng).docx",
    "ED.A009.MII.&.012.0827.docx",
    "ED.A009.MII.&.012.0828.docx",
    "ED.A009.MII.&.012.0829 (eng).docx",
    "ED.A009.MII.&.012.0829-1.docx",
    "ED.A009.MII.&.012.0830 (eng).docx",
    "ED.A009.MII.&.012.0831.docx",
    "ED.A009.MII.&.012.0832.docx",
    "ED.A009.MII.&.012.0833.docx",
    "ED.A009.MII.&.012.0834.docx",
    "ED.A009.MII.&.012.0835.docx",
    "ED.A009.MII.&.012.0836.docx",
    "ED.A009.MII.&.012.0837.docx",
    "ED.A009.MII.&.012.0839.docx",
    "ED.A009.MII.&.012.0840 (eng).docx",
    "ED.A009.MII.&.012.0841.docx",
    "ED.A009.MII.&.012.0842 (eng).docx",
    "ED.A009.MII.&.012.0843 (eng).docx",
    "ED.A009.MII.&.012.0844.docx",
    "ED.A009.MII.&.012.0845 (eng).docx",
    "ED.A009.MII.&.012.0846 (eng).docx",
    "ED.A009.MII.&.012.0847.docx",
    "ED.A009.MII.&.012.0848.docx",
    "ED.A009.MII.&.012.0849 (eng).docx",
    "ED.A009.MII.&.012.0850 (eng).docx",
    "ED.A009.MII.&.012.0852 (eng).docx",
    "ED.A009.MII.&.012.0853 (eng).docx",
    "ED.A009.MII.&.012.0854.docx",
    "ED.A009.MII.&.012.0855.docx",
    "ED.A009.MII.&.012.0856 (eng).docx",
    "ED.A009.MII.&.012.0859.docx",
    "ED.A009.MII.&.012.0860.docx",
    "ED.A009.MII.&.012.0861.docx",
    "ED.A009.MII.&.012.0863 (eng).docx",
    "ED.A009.MII.&.012.0864.docx",
    "ED.A009.MII.&.012.0865.docx",
    "ED.A009.MII.&.012.0866 (eng).docx",
    "ED.A009.MII.&.012.0867.docx",
    "ED.A009.MII.&.012.0868.docx",
    "ED.A009.MII.&.012.0869 (eng).docx",
    "ED.A009.MII.&.012.0870 (eng).docx",
    "ED.A009.MII.&.012.0871.docx",
    "ED.A009.MII.&.012.0872 (eng).docx",
    "ED.A009.MII.&.012.0873.docx",
    "ED.A009.MII.&.012.0874 (eng).docx",
    "ED.A009.MII.&.012.0875 (eng).docx",
    "ED.A009.MII.&.012.0876 (eng).docx",
    "ED.A009.MII.&.012.0877.docx",
    "ED.A009.MII.&.012.0878.docx",
    "ED.A009.MII.&.012.0879.docx",
    "ED.A009.MII.&.012.0880.docx",
    "ED.A009.MII.&.012.0881.docx",
    "ED.A009.MII.&.012.0882 (eng) от 15.05.2024 - в 12.16.docx",
    "ED.A009.MII.&.012.0884 (eng).docx",
    "ED.A009.MII.&.012.0885 (eng).docx",
    "ED.A009.MII.&.012.0886 (eng).docx",
    "ED.A009.MII.&.012.0888.docx",
    "ED.A009.MII.&.012.0889 (eng).docx",
    "ED.A009.MII.&.012.0890 (общий).docx",
    "ED.A009.MII.&.012.0891.docx",
    "ED.A009.MII.&.012.0892 (eng).docx",
    "ED.A009.MII.&.012.0893 (eng).docx",
    "ED.A009.MII.&.012.0894 (eng).docx",
    "ED.A009.MII.&.012.0895 (eng).docx",
    "ED.A009.MII.&.012.0896.docx",
    "ED.A009.MII.&.012.0897.docx",
    "ED.A009.MII.&.012.0898 (eng) от 22.05.2024 - в 15.07.docx",
    "ED.A009.MII.&.012.0899.docx",
    "ED.A009.MII.&.012.0900 (eng).docx",
    "ED.A009.MII.&.012.0901.docx",
    "ED.A009.MII.&.012.0902.docx",
    "ED.A009.MII.&.012.0903.docx",
    "ED.A009.MII.&.012.0904.docx",
    "ED.A009.MII.&.012.0905 (eng).docx",
    "ED.A009.MII.&.012.0906.docx",
    "ED.A009.MII.&.012.0907.docx",
    "ED.A009.MII.&.012.0908 (eng).docx",
    "ED.A009.MII.&.012.0909 (eng).docx",
    "ED.A009.MII.&.012.0910 (eng).docx",
    "ED.A009.MII.&.012.0911 (eng).docx",
    "ED.A009.MII.&.012.0912.docx",
    "ED.A009.MII.&.012.0914.docx",
    "ED.A009.MII.&.012.0915 (eng).docx",
    "ED.A009.MII.&.012.0916.docx",
    "ED.A009.MII.&.012.0917.docx",
    "ED.A009.MII.&.012.0918.docx",
    "ED.A009.MII.&.012.0920.docx",
    "ED.A009.MII.&.012.0921.docx",
    "ED.A009.MII.&.012.0922.docx",
    "ED.A009.MII.&.012.0923 (eng) от 05.06.2024 - в 11.38.docx",
    "ED.A009.MII.1.015.0838 (eng).docx",
    "ED.A009.MII.&.012.0923-1 (eng).docx",
    "ED.A009.MII.&.012.0924 (eng).docx",
    "ED.A009.MII.&.012.0925.docx",
    "ED.A009.MII.&.012.0926.docx",
    "ED.A009.MII.&.012.0928.docx",
    "ED.A009.MII.&.012.0929 (eng).docx",
    "ED.A009.MII.&.012.0930 (eng).docx",
    "ED.A009.MII.&.012.0931 (eng).docx",
    "ED.A009.MII.&.012.0932-1 (eng).docx",
    "ED.A009.MII.&.012.0932-2 (eng).docx",
    "ED.A009.MII.&.012.0933.docx",
    "ED.A009.MII.&.012.0934.docx",
    "ED.A009.MII.&.012.0935 (eng).docx",
    "ED.A009.MII.&.012.0936 (eng).docx",
    "ED.A009.MII.&.012.0937 (eng).docx",
    "ED.A009.MII.&.012.0938.docx",
    "ED.A009.MII.&.012.0939.docx",
    "ED.A009.MII.&.012.0940 (eng).docx",
    "ED.A009.MII.&.012.0942.docx",
    "ED.A009.MII.&.012.0943 (eng).docx",
    "ED.A009.MII.&.012.0944 (eng).docx",
    "ED.A009.MII.&.012.0945 (eng).docx",
    "ED.A009.MII.&.012.0946 (eng).docx",
    "ED.A009.MII.&.012.0947 (eng).docx",
    "ED.A009.MII.&.012.0948 (eng).docx",
    "ED.A009.MII.&.012.0949.docx",
    "ED.A009.MII.&.012.0950 (eng).docx",
    "ED.A009.MII.&.012.0951 (eng).docx",
    "ED.A009.MII.&.012.0952-1 (eng).docx",
    "ED.A009.MII.&.012.0952-2 (eng).docx",
    "ED.A009.MII.&.012.0953 (eng).docx",
    "ED.A009.MII.&.012.0954.docx",
    "ED.A009.MII.&.012.0955 (eng).docx",
    "ED.A009.MII.&.012.0956.docx",
    "ED.A009.MII.&.012.0958 (eng).docx",
    "ED.A009.MII.&.012.0959 (eng).docx",
    "ED.A009.MII.&.012.0960.docx",
    "ED.A009.MII.&.012.0961.docx",
    "ED.A009.MII.&.012.0962.docx",
    "ED.A009.MII.&.012.0963 (eng).docx",
    "ED.A009.MII.&.012.0964 (eng).docx",
    "ED.A009.MII.&.012.0965.docx",
    "ED.A009.MII.&.012.0966.docx",
    "ED.A009.MII.&.012.0967.docx",
    "ED.A009.MII.&.012.0968.docx",
    "ED.A009.MII.&.012.0969 (eng).docx",
    "ED.A009.MII.&.012.0971.docx",
    "ED.A009.MII.&.012.0972.docx",
    "ED.A009.MII.&.012.0973.docx",
    "ED.A009.MII.&.012.0974.docx",
    "ED.A009.MII.&.012.0975-1.docx",
    "ED.A009.MII.&.012.0975.docx",
    "ED.A009.MII.&.012.0976 (eng).docx",
    "ED.A009.MII.&.012.0977.docx",
    "ED.A009.MII.&.012.0978.docx",
    "ED.A009.MII.&.012.0979 (eng).docx",
    "ED.A009.MII.&.012.0980.docx",
    "ED.A009.MII.&.012.0981 (eng).docx",
    "ED.A009.MII.&.012.0982 (eng).docx",
    "ED.A009.MII.&.012.0983.docx",
    "ED.A009.MII.&.012.0984-1.docx",
    "ED.A009.MII.&.012.0984-2.docx",
    "ED.A009.MII.&.012.0985 (eng).docx",
    "ED.A009.MII.&.012.0986 (eng).docx",
    "ED.A009.MII.&.012.0987 (eng).docx",
    "ED.A009.MII.&.012.0988.docx",
    "ED.A009.MII.&.012.0989.docx",
    "ED.A009.MII.&.012.0990 (eng).docx",
    "ED.A009.MII.2.015.0919 (eng) от 04.06.2024 - в 16.20.docx",
    "ED.A009.MII.&.012.0992.docx",
    "ED.A009.MII.&.012.0993.docx",
    "ED.A009.MII.&.012.0994 (eng).docx",
    "ED.A009.MII.&.012.0995-1.docx",
    "ED.A009.MII.&.012.0995-2.docx",
    "ED.A009.MII.&.012.0996 (eng) от 23.07.2024 - в 10.05.docx",
    "ED.A009.MII.&.012.0998.docx",
    "ED.A009.MII.&.012.0999.docx",
    "ED.A009.MII.&.012.1000.docx",
    "ED.A009.MII.&.012.1001 (eng).docx",
    "ED.A009.MII.&.012.1003 (eng).docx",
    "ED.A009.MII.&.012.1004.docx",
    "ED.A009.MII.&.012.1005.docx",
    "ED.A009.MII.&.012.1006 (eng).docx",
    "ED.A009.MII.&.012.1007 (eng).docx",
    "ED.A009.MII.&.012.1008.docx",
    "ED.A009.MII.&.012.1009.docx",
    "ED.A009.MII.&.012.1010 (eng).docx",
    "ED.A009.MII.&.012.1011 (eng).docx",
    "ED.A009.MII.&.012.1012 (eng).docx",
    "ED.A009.MII.&.012.1013 (eng).docx",
    "ED.A009.MII.&.012.1014 (eng).docx",
    "ED.A009.MII.&.012.1015 (eng).docx",
    "ED.A009.MII.&.012.1016.docx",
    "ED.A009.MII.&.012.1017.docx",
    "ED.A009.MII.&.012.1018.docx",
    "ED.A009.MII.&.012.1019.docx",
    "ED.A009.MII.&.012.1020.docx",
    "ED.A009.MII.&.012.1021.docx",
    "ED.A009.MII.&.012.1022.docx",
    "ED.A009.MII.&.012.1023 (eng).docx",
    "ED.A009.MII.&.012.1024.docx",
    "ED.A009.MII.&.012.1025.docx",
    "ED.A009.MII.&.012.1026 (eng).docx",
    "ED.A009.MII.&.012.1027 (eng).docx",
    "ED.A009.MII.&.012.1028 (eng).docx",
    "ED.A009.MII.&.012.1029 (eng).docx",
    "ED.A009.MII.&.012.1030.docx",
    "ED.A009.MII.&.012.1031 (eng).docx",
    "ED.A009.MII.&.012.1032 (eng).docx",
    "ED.A009.MII.&.012.1033 (eng).docx",
    "ED.A009.MII.&.012.1034 (eng).docx",
    "ED.A009.MII.&.012.1035 (eng).docx",
    "ED.A009.MII.&.012.1036 (eng).docx",
    "ED.A009.MII.&.012.1037 (eng).docx",
    "ED.A009.MII.&.012.1038 (eng).docx",
    "ED.A009.MII.&.012.1039 (eng).docx",
    "ED.A009.MII.&.012.1040 (eng).docx",
    "ED.A009.MII.&.012.1041.docx",
    "ED.A009.MII.&.012.1042.docx",
    "ED.A009.MII.&.012.1043.docx",
    "ED.A009.MII.&.012.1044.docx",
    "ED.A009.MII.&.012.1045.docx",
    "ED.A009.MII.&.012.1046 (eng).docx",
    "ED.A009.MII.&.012.1047 (eng).docx",
    "ED.A009.MII.&.012.1048 (eng).docx",
    "ED.A009.MII.&.012.1049.docx",
    "ED.A009.MII.&.015.0997 (eng) от 28.07.2024 - в 10.47.docx",
    "ED.A009.MII.&.012.1050 (испр даты и договора).docx",
    "ED.A009.MII.&.012.1051.docx",
    "ED.A009.MII.&.012.1052 (eng).docx",
    "ED.A009.MII.&.012.1053 (eng).docx",
    "ED.A009.MII.&.012.1054 (eng).docx",
    "ED.A009.MII.&.012.1055 (eng).docx",
    "ED.A009.MII.&.012.1056 (eng).docx",
    "ED.A009.MII.&.012.1057 (eng).docx",
    "ED.A009.MII.&.012.1058 (eng).docx",
    "ED.A009.MII.&.012.1059.docx",
    "ED.A009.MII.&.012.1060 (eng).docx",
    "ED.A009.MII.&.012.1061 (испр. дата и договор).docx",
    "ED.A009.MII.&.012.1062 (eng).docx",
    "ED.A009.MII.&.012.1063 (eng).docx",
    "ED.A009.MII.&.012.1065.docx",
    "ED.A009.MII.&.012.1066 (испр. дата и договор).docx",
    "ED.A009.MII.&.012.1067.docx",
    "ED.A009.MII.&.012.1068.docx",
    "ED.A009.MII.&.012.1069.docx",
    "ED.A009.MII.&.012.1070.docx",
    "ED.A009.MII.&.012.1073 (eng).docx",
    "ED.A009.MII.&.012.1074 (eng).docx",
    "ED.A009.MII.&.012.1075 (eng).docx",
    "ED.A009.MII.&.012.1077.docx",
    "ED.A009.MII.&.012.1078 (eng).docx",
    "ED.A009.MII.&.012.1081.docx",
    "ED.A009.MII.&.012.1082 (eng).docx",
    "ED.A009.MII.&.012.1083 (eng).docx",
    "ED.A009.MII.&.012.1085 (eng).docx",
    "ED.A009.MII.&.012.1086.docx",
    "ED.A009.MII.&.012.1087 (eng).docx",
    "ED.A009.MII.&.012.1088 (eng).docx",
    "ED.A009.MII.&.012.1089 (eng).docx",
    "ED.A009.MII.&.012.1090.docx",
    "ED.A009.MII.&.012.1091 (eng).docx",
    "ED.A009.MII.&.012.1092 (eng).docx",
    "ED.A009.MII.&.012.1093.docx",
    "ED.A009.MII.&.012.1094.docx",
    "ED.A009.MII.&.012.1095 (eng).docx",
    "ED.A009.MII.&.012.1096.docx",
    "ED.A009.MII.&.012.1097.docx",
    "ED.A009.MII.&.012.1098.docx",
    "ED.A009.MII.&.012.1099 (eng).docx",
    "ED.A009.MII.&.012.1101.docx",
    "ED.A009.MII.&.012.1102 (eng).docx",
    "ED.A009.MII.&.012.1103 (eng).docx",
    "ED.A009.MII.&.012.1104.docx",
    "ED.A009.MII.&.012.1105.docx",
    "ED.A009.MII.&.012.1106.docx",
    "ED.A009.MII.&.012.1108 (eng).docx",
    "ED.A009.MII.&.012.1109.docx",
    "ED.A009.MII.&.012.1110.docx",
    "ED.A009.MII.&.012.1111.docx",
    "ED.A009.MII.&.012.1112.docx",
    "ED.A009.MII.&.012.1113.docx",
    "ED.A009.MII.&.012.1114 (eng).docx",
    "ED.A009.MII.&.012.1115.docx",
    "ED.A009.MII.&.012.1116.docx",
    "ED.A009.MII.&.012.1117.docx",
    "ED.A009.MII.&.012.1119.docx",
    "ED.A009.MII.&.012.1120.docx",
    "ED.A009.MII.&.012.1121.docx",
    "ED.A009.MII.&.012.1122.docx",
    "ED.A009.MII.&.012.1124.docx",
    "ED.A009.MII.&.012.1125 (испр. даты и договора).docx",
    "ED.A009.MII.&.012.1127.docx",
    "ED.A009.MII.&.012.1128.docx",
    "ED.A009.MII.&.012.1130.docx",
    "ED.A009.MII.&.012.1131.docx",
    "ED.A009.MII.&.012.1132.docx",
    "ED.A009.MII.&.012.1133 (eng).docx",
    "ED.A009.MII.&.012.1134.docx",
    "ED.A009.MII.&.012.1135.docx",
    "ED.A009.MII.&.012.1136.docx",
    "ED.A009.MII.&.012.1137.docx",
    "ED.A009.MII.&.012.1139.docx",
    "ED.A009.MII.&.012.1140.docx",
    "ED.A009.MII.&.012.1141.docx",
    "ED.A009.MII.&.012.1142.docx",
    "ED.A009.MII.&.012.1143.docx",
    "ED.A009.MII.&.012.1145 (eng).docx",
    "ED.A009.MII.&.012.1146.docx",
    "ED.A009.MII.&.012.1147 не актуально.docx",
    "ED.A009.MII.&.012.1148.docx",
    "ED.A009.MII.&.012.1149.docx",
    "ED.A009.MII.&.012.1150 (eng).docx",
    "ED.A009.MII.&.012.1151 (eng).docx",
    "ED.A009.MII.&.012.1153 (eng).docx",
    "ED.A009.MII.&.012.1154 (eng).docx",
    "ED.A009.MII.&.012.1155 (eng).docx",
    "ED.A009.MII.&.012.1156.docx",
    "ED.A009.MII.&.012.1157.docx",
    "ED.A009.MII.&.012.1158.docx",
    "ED.A009.MII.&.012.1160 (eng).docx",
    "ED.A009.MII.&.012.1161.docx",
    "ED.A009.MII.&.012.1162.docx",
    "ED.A009.MII.&.012.1164 (eng).docx",
    "ED.A009.MII.&.012.1165 (eng).docx",
    "ED.A009.MII.&.012.1166 (eng).docx",
    "ED.A009.MII.&.012.1167.docx",
    "ED.A009.MII.&.012.1168.docx",
    "ED.A009.MII.&.012.1169.docx",
    "ED.A009.MII.&.012.1170.docx",
    "ED.A009.MII.&.012.1171 (eng).docx",
    "ED.A009.MII.&.012.1172 (eng).docx",
    "ED.A009.MII.&.012.1173.docx",
    "ED.A009.MII.&.012.1174.docx",
    "ED.A009.MII.&.012.1175 (eng).docx",
    "ED.A009.MII.&.012.1176.docx",
    "ED.A009.MII.&.012.1178.docx",
    "ED.A009.MII.&.012.1179.docx",
    "ED.A009.MII.&.012.1180 (eng).docx",
    "ED.A009.MII.&.012.1181.docx",
    "ED.A009.MII.&.012.1182.docx",
    "ED.A009.MII.&.012.1183 (eng).docx",
    "ED.A009.MII.&.012.1184.docx",
    "ED.A009.MII.&.012.1185 (eng).docx",
    "ED.A009.MII.&.012.1186.docx",
    "ED.A009.MII.&.012.1187.docx",
    "ED.A009.MII.&.012.1188.docx",
    "ED.A009.MII.&.012.1189 (eng).docx",
    "ED.A009.MII.&.012.1190.docx",
    "ED.A009.MII.&.012.1191.docx",
    "ED.A009.MII.&.012.1192.docx",
    "ED.A009.MII.&.012.1193 (eng).docx",
    "ED.A023.MII.&.&&&.01195-II.docx",
    "ED.A023.MII.&.&&&.01196-II.docx",
    "ED.A023.MII.&.&&&.01197-II.docx",
    "ED.A023.MII.&.&&&.01198-II.docx",
    "ED.A023.MII.&.&&&.01199-II.docx",
    "ED.A023.MII.&.&&&.01200-II (eng).docx",
    "ED.A023.MII.&.&&&.01201-II.docx",
    "ED.A023.MII.&.&&&.01202-II — копия.docx",
    "ED.A023.MII.&.&&&.01203-II (eng).docx",
    "ED.A023.MII.&.&&&.01204-II (eng).docx",
    "ED.A023.MII.&.&&&.01205-II.docx",
    "ED.A023.MII.&.&&&.01206-II.docx",
    "ED.A023.MII.&.&&&.01207-II.docx",
    "ED.A023.MII.&.&&&.01208-II.docx",
    "ED.A023.MII.&.&&&.01210-II (eng).docx",
    "ED.A023.MII.&.&&&.01211-II (eng).docx",
    "ED.A023.MII.&.&&&.01212-II (eng).docx",
    "ED.A023.MII.&.&&&.01213-II (eng) с испр. датой.docx",
    "ED.A023.MII.&.&&&.01214-II (eng).docx",
    "ED.A023.MII.&.&&&.01215-II (eng).docx",
    "ED.A023.MII.&.&&&.01216-II.docx",
    "ED.A023.MII.&.&&&.01217-II.docx",
    "ED.A023.MII.&.&&&.01218-II.docx",
    "ED.A023.MII.&.&&&.01219-II (eng).docx",
    "ED.A023.MII.&.&&&.01220-II.docx",
    "ED.A023.MII.&.&&&.01221-II.docx",
    "ED.A023.MII.&.&&&.01223-II.docx",
    "ED.A023.MII.&.&&&.01224-II.docx",
    "ED.A023.MII.&.&&&.01225-II.docx",
    "ED.A023.MII.&.&&&.01226-II.docx",
    "ED.A023.MII.&.&&&.01227-II (eng).docx",
    "ED.A023.MII.&.&&&.01228-II.docx",
    "ED.A023.MII.&.&&&.01229-II (eng).docx",
    "ED.A023.MII.&.&&&.01230-II (eng).docx",
    "ED.A023.MII.&.&&&.01231-II.docx",
    "ED.A023.MII.&.&&&.01232-II.docx",
    "ED.A023.MII.&.&&&.01233-II.docx",
    "ED.A023.MII.&.&&&.01234-II.docx",
    "ED.A023.MII.&.&&&.01235-II (eng).docx",
    "ED.A023.MII.&.&&&.01236-II (eng).docx",
    "ED.A023.MII.&.&&&.01237-II.docx",
    "ED.A023.MII.&.&&&.01238-II.docx",
    "ED.A023.MII.&.&&&.01239-II (eng).docx",
    "ED.A023.MII.&.&&&.01240-II.docx",
    "ED.A023.MII.&.&&&.01241-II (eng).docx",
    "ED.A023.MII.&.&&&.01242-II.docx",
    "ED.A023.MII.&.&&&.01243-II.docx",
    "ED.A023.MII.&.&&&.01244-II (eng).docx",
    "ED.A023.MII.&.&&&.01245-II.docx",
    "ED.A023.MII.&.&&&.01246-II.docx",
    "ED.A023.MII.&.&&&.01247-II.docx",
    "ED.A023.MII.&.&&&.01248-II.docx",
    "ED.A023.MII.&.&&&.01249-II (eng).docx",
    "ED.A023.MII.&.&&&.01250-II (eng).docx",
    "ED.A023.MII.&.&&&.01251-II.docx",
    "ED.A023.MII.&.&&&.01252-II.docx",
    "ED.A023.MII.&.&&&.01253-II (eng).docx",
    "ED.A023.MII.&.&&&.01254-II (eng).docx",
    "ED.A023.MII.&.&&&.01255-II.docx",
    "ED.A023.MII.&.&&&.01256-II.docx",
    "ED.A023.MII.&.&&&.01257-II (eng).docx",
    "ED.A023.MII.&.&&&.01258-II (eng).docx",
    "ED.A023.MII.&.&&&.01259-II.docx",
    "ED.A023.MII.&.&&&.01260-II.docx",
    "ED.A023.MII.&.&&&.01261-II.docx",
    "ED.A023.MII.&.&&&.01262-II.docx",
    "ED.A023.MII.&.&&&.01263-II.docx",
    "ED.A023.MII.&.&&&.01264-II (eng).docx",
    "ED.A023.MII.&.&&&.01265-II (eng).docx",
    "ED.A023.MII.&.&&&.01266-II (eng) положит.docx",
    "ED.A023.MII.&.&&&.01267-II.docx",
    "ED.A023.MII.&.&&&.01268-II.docx",
    "ED.A023.MII.&.&&&.01269-II.docx",
    "ED.A023.MII.&.&&&.01270-II (eng).docx",
    "ED.A023.MII.&.&&&.01271-II.docx",
    "ED.A023.MII.&.&&&.01272-II.docx",
    "ED.A023.MII.&.&&&.01273-II.docx",
    "ED.A023.MII.&.&&&.01274-II (eng) с испр. датой.docx",
    "ED.A023.MII.&.&&&.01276-II.docx",
    "ED.A023.MII.&.&&&.01277-II.docx",
    "ED.A023.MII.&.&&&.01278-II.docx",
    "ED.A023.MII.&.&&&.01280-II (eng).docx",
    "ED.A023.MII.&.&&&.01281-II.docx",
    "ED.A023.MII.&.&&&.01282-II (eng).docx",
    "ED.A023.MII.&.&&&.01283-II (eng).docx",
    "ED.A023.MII.&.&&&.01284-II.docx",
    "ED.A023.MII.&.&&&.01285-II (eng).docx",
    "ED.A023.MII.&.&&&.01286-II (eng).docx",
    "ED.A023.MII.&.&&&.01287-II (eng).docx",
    "ED.A023.MII.&.&&&.01288-II (eng).docx",
    "ED.A023.MII.&.&&&.01289-II (eng).docx",
    "ED.A023.MII.&.&&&.01290-II.docx",
    "ED.A023.MII.&.&&&.01291-II (eng).docx",
    "ED.A023.MII.&.&&&.01292-II.docx",
    "ED.A023.MII.&.&&&.01293-II.docx",
    "ED.A023.MII.&.&&&.01294-II (eng).docx",
    "ED.A023.MII.&.&&&.01295-II (eng).docx",
    "ED.A023.MII.&.&&&.01296-II (eng).docx",
    "ED.A023.MII.&.&&&.01297-II.docx",
    "ED.A023.MII.&.&&&.01299-II (eng).docx",
    "ED.A023.MII.&.&&&.01300-II (eng).docx",
    "ED.A023.MII.&.&&&.01301-II.docx",
    "ED.A023.MII.&.&&&.01302-II.docx",
    "ED.A023.MII.&.&&&.01303-II.docx",
    "ED.A023.MII.&.&&&.01304-II.docx",
    "ED.A023.MII.&.&&&.01305-II (eng).docx",
    "ED.A023.MII.&.&&&.01306-II (eng).docx",
    "ED.A023.MII.&.&&&.01307-II.docx",
    "ED.A023.MII.&.&&&.01308-II.docx",
    "ED.A023.MII.&.&&&.01309-II (eng).docx",
    "ED.A023.MII.&.&&&.01310-II (eng).docx",
    "ED.A023.MII.&.&&&.01311-II.docx",
    "ED.A023.MII.&.&&&.01313-II.docx",
    "ED.A023.MII.&.&&&.01314-II.docx",
    "ED.A023.MII.&.&&&.01315-II.docx",
    "ED.A023.MII.&.&&&.01316-II (eng).docx",
    "ED.A023.MII.&.&&&.01317-II.docx",
    "ED.A023.MII.&.&&&.01318-II (eng).docx",
    "ED.A023.MII.&.&&&.01319-II (eng).docx",
    "ED.A023.MII.&.&&&.01320-II (eng).docx",
    "ED.A023.MII.&.&&&.01321-II (eng).docx",
    "ED.A023.MII.&.&&&.01322-II.docx",
    "ED.A023.MII.&.&&&.01323-II.docx",
    "ED.A023.MII.&.&&&.01325-II.docx",
    "ED.A023.MII.&.&&&.01326-II (eng).docx",
    "ED.A023.MII.&.&&&.01327-II.docx",
    "ED.A023.MII.&.&&&.01328-II (eng).docx",
    "ED.A023.MII.&.&&&.01329-II.docx",
    "ED.A023.MII.&.&&&.01330-II (eng).docx",
    "ED.A023.MII.&.&&&.01331-II (eng).docx",
    "ED.A023.MII.&.&&&.01332-II (eng).docx",
    "ED.A023.MII.&.&&&.01333-II (eng).docx",
    "ED.A023.MII.&.&&&.01334-II (eng).docx",
    "ED.A023.MII.&.&&&.01336-II.docx",
    "ED.A023.MII.&.&&&.01337-II (eng).docx",
    "ED.A023.MII.&.&&&.01338-II (eng).docx",
    "ED.A023.MII.&.&&&.01339-II (eng).docx",
    "ED.A023.MII.&.&&&.01340-II.docx",
    "ED.A023.MII.&.&&&.01341-II.docx",
    "ED.A023.MII.&.&&&.01342-II (eng).docx",
    "ED.A023.MII.&.&&&.01343-II (eng).docx",
    "ED.A023.MII.&.&&&.01344-II.docx",
    "ED.A023.MII.&.&&&.01345-II (eng).docx",
    "ED.A023.MII.&.&&&.01346-II.docx",
    "ED.A023.MII.&.&&&.01347-II (eng).docx",
    "ED.A023.MII.&.&&&.01348-II.docx",
    "ED.A023.MII.&.&&&.01349-II.docx",
    "ED.A023.MII.&.&&&.01350-II (eng).docx",
    "ED.A023.MII.&.&&&.01351-II.docx",
    "ED.A023.MII.&.&&&.01352-II (eng).docx",
    "ED.A023.MII.&.&&&.01353-II.docx",
    "ED.A023.MII.&.&&&.01354-II (eng).docx",
    "ED.A023.MII.&.&&&.01355-II (eng).docx",
    "ED.A023.MII.&.&&&.01356-II (eng).docx",
    "ED.A023.MII.&.&&&.01357-II.docx",
    "ED.A023.MII.&.&&&.01358-II (eng).docx",
    "ED.A023.MII.&.&&&.01359-II (eng).docx",
    "ED.A023.MII.&.&&&.01360-II (eng).docx",
    "ED.A023.MII.&.&&&.01361-II.docx",
    "ED.A023.MII.&.&&&.01362-II.docx",
    "ED.A023.MII.&.&&&.01363-II (eng).docx",
    "ED.A023.MII.&.&&&.01364-II (eng).docx",
    "ED.A023.MII.&.&&&.01365-II (eng).docx",
    "ED.A023.MII.&.&&&.01366-II.docx",
    "ED.A023.MII.&.&&&.01367-II.docx",
    "ED.A023.MII.&.&&&.01368-II.docx",
    "ED.A023.MII.&.&&&.01370-II (eng).docx",
    "ED.A023.MII.&.&&&.01371-II (eng).docx",
    "ED.A023.MII.&.&&&.01372-II (eng).docx",
    "ED.A023.MII.&.&&&.01373-II (eng).docx",
    "ED.A023.MII.&.&&&.01374-II (eng).docx",
    "ED.A023.MII.&.&&&.01375-II.docx",
    "ED.A023.MII.&.&&&.01377-1-II (eng).docx",
    "ED.A023.MII.&.&&&.01377-2-II (eng).docx",
    "ED.A023.MII.&.&&&.01378-II (eng).docx",
    "ED.A023.MII.&.&&&.01379-II.docx",
    "ED.A023.MII.&.&&&.01380-II (eng).docx",
    "ED.A023.MII.&.&&&.01381-II (eng).docx",
    "ED.A023.MII.&.&&&.01382-II (eng).docx",
    "ED.A023.MII.&.&&&.01383-II (eng).docx",
    "ED.A023.MII.&.&&&.01384-II (eng).docx",
    "ED.A023.MII.&.&&&.01385-II.docx",
    "ED.A023.MII.&.&&&.01386-II (eng).docx",
    "ED.A023.MII.&.&&&.01387-II.docx",
    "ED.A023.MII.&.&&&.01388-II.docx",
    "ED.A023.MII.&.&&&.01389-II (eng).docx",
    "ED.A023.MII.&.&&&.01390-II (eng).docx",
    "ED.A023.MII.&.&&&.01391-II.docx",
    "ED.A023.MII.&.&&&.01392-II (eng).docx",
    "ED.A023.MII.&.&&&.01393-II (eng).docx",
    "ED.A023.MII.&.&&&.01394-II (eng).docx",
    "ED.A023.MII.&.&&&.01395-II.docx",
    "ED.A023.MII.&.&&&.01395.1-II.docx",
    "ED.A023.MII.&.&&&.01397-II.docx",
    "ED.A023.MII.&.&&&.01398-II.docx",
    "ED.A023.MII.&.&&&.01399-II.docx",
    "ED.A023.MII.&.&&&.01401-II (eng).docx",
    "ED.A023.MII.&.&&&.01402-II (eng).docx",
    "ED.A023.MII.&.&&&.01403-II (eng).docx",
    "ED.A023.MII.&.&&&.01404-II.docx",
    "ED.A023.MII.&.&&&.01405-II (eng).docx",
    "ED.A023.MII.&.&&&.01406-II (eng).docx",
    "ED.A023.MII.&.&&&.01407-II (eng).docx",
    "ED.A023.MII.&.&&&.01408-II.docx",
    "ED.A023.MII.&.&&&.01410-II (eng).docx",
    "ED.A023.MII.&.&&&.01411-II (eng).docx",
    "ED.A023.MII.&.&&&.01412-II (eng).docx",
    "ED.A023.MII.&.&&&.01413-II (eng).docx",
    "ED.A023.MII.&.&&&.01414-II.docx",
    "ED.A023.MII.&.&&&.01415-II (eng).docx",
    "ED.A023.MII.&.&&&.01416-II (eng).docx",
    "ED.A023.MII.&.&&&.01417-II (eng).docx",
    "ED.A023.MII.&.&&&.01418-II.docx",
    "ED.A023.MII.&.&&&.01419-II (eng).docx",
    "ED.A023.MII.&.&&&.01420-II.docx",
    "ED.A023.MII.&.&&&.01420-1-II (eng).docx",
    "ED.A023.MII.&.&&&.01422-II (eng).docx",
    "ED.A023.MII.&.&&&.01423-II (eng).docx",
    "ED.A023.MII.&.&&&.01424-II.docx",
    "ED.A023.MII.&.&&&.01425-II.docx",
    "ED.A023.MII.&.&&&.01426-II.docx",
    "ED.A023.MII.&.&&&.01427-II (eng).docx",
    "ED.A023.MII.&.&&&.01428-II (eng).docx",
    "ED.A023.MII.&.&&&.01429-II.docx",
    "ED.A023.MII.&.&&&.01430-II (eng)  с испр. датой.docx",
    "ED.A023.MII.&.&&&.01431-II.docx",
    "ED.A023.MII.&.&&&.01432-II.docx",
    "ED.A023.MII.&.&&&.01433-II.docx",
    "ED.A023.MII.&.&&&.01434-II (eng).docx",
    "ED.A023.MII.&.&&&.01435-II.docx",
    "ED.A023.MII.&.&&&.01436-II.docx",
    "ED.A023.MII.&.&&&.01437-II (eng).docx",
    "ED.A023.MII.&.&&&.01439-II.docx",
    "ED.A023.MII.&.&&&.01440-II (eng).docx",
    "ED.A023.MII.&.&&&.01441-II.docx",
    "ED.A023.MII.&.&&&.01442-II (eng) с испр. датой.docx",
    "ED.A023.MII.&.&&&.01443-II (eng).docx",
    "ED.A023.MII.&.&&&.01444-II (eng).docx",
    "ED.A023.MII.&.&&&.01445-II (eng).docx",
    "ED.A023.MII.&.&&&.01446-II.docx",
    "ED.A023.MII.&.&&&.01447-II.docx",
    "ED.A023.MII.&.&&&.01448-II (eng).docx",
    "ED.A023.MII.&.&&&.01449-II (eng).docx",
    "ED.A023.MII.&.&&&.01450-II.docx",
    "ED.A023.MII.&.&&&.01451-II.docx",
    "ED.A023.MII.&.&&&.01452-II.docx",
    "ED.A023.MII.&.&&&.01453-II.docx",
    "ED.A023.MII.&.&&&.01454-II АСЭ.docx",
    "ED.A023.MII.&.&&&.01455-II (с изм. датой).docx",
    "ED.A023.MII.&.&&&.01457-II (eng).docx",
    "ED.A023.MII.&.&&&.01458-II (eng).docx",
    "ED.A023.MII.&.&&&.01459-II (eng).docx",
    "ED.A023.MII.&.&&&.01460-II (eng) от 11.06.2025 - в 15.02.docx",
    "ED.A023.MII.&.&&&.01461-II (eng).docx",
    "ED.A023.MII.&.&&&.01462-II.docx",
    "ED.A023.MII.&.&&&.01463-II (eng).docx",
    "ED.A023.MII.&.&&&.01464-II (eng).docx",
    "ED.A023.MII.&.&&&.01465-II (eng).docx",
    "ED.A023.MII.&.&&&.01466-II Изменение даты.docx",
    "ED.A023.MII.&.&&&.01467-II.docx",
    "ED.A023.MII.&.&&&.01468-II (eng).docx",
    "ED.A023.MII.&.&&&.01469-II (eng).docx",
    "ED.A023.MII.&.&&&.01470-II (eng) с испр. датой (01.07.2025).docx",
    "ED.A023.MII.&.&&&.01471-II (eng).docx",
    "ED.A023.MII.&.&&&.01472-II (eng).docx",
    "ED.A023.MII.&.&&&.01473-II (eng).docx",
    "ED.A023.MII.&.&&&.01474-II (eng).docx",
    "ED.A023.MII.&.&&&.01476-II.docx",
    "ED.A023.MII.&.&&&.01477-II.docx",
    "ED.A023.MII.&.&&&.01478-II.docx",
    "ED.A023.MII.&.&&&.01479-II.docx",
    "ED.A023.MII.&.&&&.01482-II.docx",
    "ED.A023.MII.&.&&&.01483-II (eng).docx",
    "ED.A023.MII.&.&&&.01485-II (eng).docx",
    "ED.A023.MII.&.&&&.01486-II.docx",
    "ED.A023.MII.&.&&&.01487-II.docx",
    "ED.A023.MII.&.&&&.01488-II.docx",
    "ED.A023.MII.&.&&&.01489-II (eng).docx",
    "ED.A023.MII.&.&&&.01491-II (eng).docx",
    "ED.A023.MII.&.&&&.01492-II (eng).docx",
    "ED.A023.MII.&.&&&.01493-II (eng).docx",
    "ED.A023.MII.&.&&&.01494-II (eng).docx",
    "ED.A023.MII.&.&&&.01495-II.docx",
    "ED.A023.MII.&.&&&.01496-II.docx",
    "ED.A023.MII.&.&&&.01497-II (eng).docx",
    "ED.A023.MII.&.&&&.01498-II.docx",
    "ED.A023.MII.&.&&&.01499-II.docx",
    "ED.A023.MII.&.&&&.01500-II (eng).docx",
    "ED.A023.MII.&.&&&.01501-II.docx",
    "ED.A023.MII.&.&&&.01502-II.docx",
    "ED.A023.MII.&.&&&.01503-II.docx",
    "ED.A023.MII.&.&&&.01504-II.docx",
    "ED.A023.MII.&.&&&.01505-II.docx",
    "ED.A023.MII.&.&&&.01506-II.docx",
    "ED.A023.MII.&.&&&.01507-II.docx",
    "ED.A023.MII.&.&&&.01508-II.docx",
    "ED.A023.MII.&.&&&.01509-II.docx",
    "ED.A023.MII.&.&&&.01510-II (испр).docx",
    "ED.A023.MII.&.&&&.01511-II (eng).docx",
    "ED.A023.MII.&.&&&.01512-II (eng).docx",
    "ED.A023.MII.&.&&&.01513-II.docx",
    "ED.A023.MII.&.&&&.01514-II (eng).docx",
    "ED.A023.MII.&.&&&.01515-II (eng).docx",
    "ED.A023.MII.&.&&&.01516-II (eng).docx",
    "ED.A023.MII.&.&&&.01517-II (eng).docx",
    "ED.A023.MII.&.&&&.01518-II.docx",
    "ED.A023.MII.&.&&&.01518.1-II (eng).docx",
    "ED.A023.MII.&.&&&.01519-II.docx",
    "ED.A023.MII.&.&&&.01521-II.docx",
    "ED.A023.MII.&.&&&.01522-II.docx",
    "ED.A023.MII.&.&&&.01523-II.docx",
    "ED.A023.MII.&.&&&.01524-II.docx",
    "ED.A023.MII.&.&&&.01525-II (eng).docx",
    "ED.A023.MII.&.&&&.01526-II.docx",
    "ED.A023.MII.&.&&&.01527-II (eng).docx",
    "ED.A023.MII.&.&&&.01529-II.docx",
    "ED.A023.MII.&.&&&.01530-II (eng).docx",
    "ED.A023.MII.&.&&&.01531-II (eng).docx",
    "ED.A023.MII.&.&&&.01532-II.docx",
    "ED.A023.MII.&.&&&.01533-II.docx",
    "ED.A023.MII.&.&&&.01534-II.docx",
    "ED.A023.MII.&.&&&.01536-II (с изм. даты).docx",
    "ED.A023.MII.&.&&&.01537-II.docx",
    "ED.A023.MII.&.&&&.01538-II.docx",
    "ED.A023.MII.&.&&&.01539-II (eng).docx",
    "ED.A023.MII.&.&&&.01540-II (eng).docx",
    "ED.A023.MII.&.&&&.01541-II (eng).docx",
    "ED.A023.MII.&.&&&.01542-II (eng).docx",
    "ED.A023.MII.&.&&&.01543-II (eng).docx",
    "ED.A023.MII.&.&&&.01544-II (eng).docx",
    "ED.A023.MII.&.&&&.01545-II (eng).docx",
    "ED.A023.MII.&.&&&.01546-II (eng).docx",
    "ED.A023.MII.&.&&&.01547-II (eng).docx",
    "ED.A023.MII.&.&&&.01548-II (eng).docx",
    "ED.A023.MII.&.&&&.01549-II (eng).docx",
    "ED.A023.MII.&.&&&.01550-II (eng).docx",
    "ED.A023.MII.&.&&&.01551-II.docx",
    "ED.A023.MII.&.&&&.01552-II.docx",
    "ED.A023.MII.&.&&&.01553-II.docx",
    "ED.A023.MII.&.&&&.01554-II.docx",
    "ED.A023.MII.&.&&&.01555-II (eng).docx",
    "ED.A023.MII.&.&&&.01556-II (eng).docx",
    "ED.A023.MII.&.&&&.01557-II (eng).docx",
    "ED.A023.MII.&.&&&.01558-II (eng).docx",
    "ED.A023.MII.&.&&&.01559-II.docx",
    "ED.A023.MII.&.&&&.01560-II.docx",
    "ED.A023.MII.&.&&&.01561-II.docx",
    "ED.A023.MII.&.&&&.01562-II.docx",
    "ED.A023.MII.&.&&&.01565-II.docx",
    "ED.A023.MII.&.&&&.01566-II (eng).docx",
    "ED.A023.MII.&.&&&.01567-II (eng).docx",
    "ED.A023.MII.&.&&&.01568-II.docx",
    "ED.A023.MII.&.&&&.01569-II (eng).docx",
    "ED.A023.MII.&.&&&.01570-II (eng).docx",
    "ED.A023.MII.&.&&&.01571-II.docx",
    "ED.A023.MII.&.&&&.01572-II (eng).docx",
    "ED.A023.MII.&.&&&.01573-II (eng).docx",
    "ED.A023.MII.&.&&&.01574-II.docx",
    "ED.A023.MII.&.&&&.01575-II.docx",
    "ED.A023.MII.&.&&&.01576-II.docx",
    "ED.A023.MII.&.&&&.01577-II (eng).docx",
    "ED.A023.MII.&.&&&.01578-II (eng).docx",
    "ED.A023.MII.&.&&&.01580-II (eng).docx",
    "ED.A023.MII.&.&&&.01581-II (eng).docx",
    "ED.A023.MII.&.&&&.01582-II (eng).docx",
    "ED.A023.MII.&.&&&.01583-II.docx",
    "ED.A023.MII.&.&&&.01584-II (eng).docx",
    "ED.A023.MII.&.&&&.01585-II (eng).docx",
    "ED.A023.MII.&.&&&.01586-II.docx",
    "ED.A023.MII.&.&&&.01587-II.docx",
    "ED.A023.MII.&.&&&.01588-II (eng).docx",
    "ED.A023.MII.&.&&&.01590-II.docx",
    "ED.A023.MII.&.&&&.01591-II.docx",
    "ED.A023.MII.&.&&&.01592-II.docx",
    "ED.A023.MII.&.&&&.01594-II (eng).docx",
    "ED.A023.MII.&.&&&.01595-II.docx",
    "ED.A023.MII.&.&&&.01596-II.docx",
    "ED.A023.MII.&.&&&.01597-II (eng).docx",
    "ED.A023.MII.&.&&&.01598-II (eng).docx",
    "ED.A023.MII.&.&&&.01599-II (eng).docx",
    "ED.A023.MII.&.&&&.01600-II.docx",
    "ED.A023.MII.&.&&&.01601-II (eng).docx",
    "ED.A023.MII.&.&&&.01602-II.docx",
    "ED.A023.MII.&.&&&.01603-II.docx",
    "ED.A023.MII.&.&&&.01609-II (eng).docx",
    "ED.A023.MII.&.&&&.01610-II.docx",
    "ED.A023.MII.&.&&&.01611-II.docx",
    "ED.A023.MII.&.&&&.01612-II.docx",
    "ED.A023.MII.&.&&&.01613-II.docx",
    "ED.A023.MII.&.&&&.01613-1-II (eng).docx",
    "ED.A023.MII.&.&&&.01614-II.docx",
    "ED.A023.MII.&.&&&.01615-II.docx",
    "ED.A023.MII.&.&&&.01616-II (eng).docx",
    "ED.A023.MII.&.&&&.01617-II.docx",
    "ED.A023.MII.&.&&&.01617-1-II (eng).docx",
    "ED.A023.MII.&.&&&.01618-II.docx",
    "ED.A023.MII.&.&&&.01619-II (eng).docx",
    "ED.A023.MII.&.&&&.01620-II (eng).docx",
    "ED.A023.MII.&.&&&.01621-II (eng).docx",
    "ED.A023.MII.&.&&&.01622-II (eng).docx",
    "ED.A023.MII.&.&&&.01623-II (eng).docx",
    "ED.A023.MII.&.&&&.01624-II (eng).docx",
    "ED.A023.MII.&.&&&.01625-II (eng).docx",
    "ED.A023.MII.&.&&&.01626-II.docx",
    "ED.A023.MII.&.&&&.01627-II (eng).docx",
    "ED.A023.MII.&.&&&.01628-II.docx",
    "ED.A023.MII.&.&&&.01629-II (eng).docx",
    "ED.A023.MII.&.&&&.01630-II.docx",
    "ED.A023.MII.&.&&&.01631-II (eng) ВК +.docx",
    "ED.A023.MII.&.&&&.01632-II (eng).docx",
    "ED.A023.MII.&.&&&.01633-II (eng).docx",
    "ED.A023.MII.&.&&&.01634-II (eng).docx",
    "ED.A023.MII.&.&&&.01635-II (eng).docx",
    "ED.A023.MII.&.&&&.01636-II.docx",
    "ED.A023.MII.&.&&&.01637-II.docx",
    "ED.A023.MII.&.&&&.01638-II.docx",
    "ED.A023.MII.&.&&&.01639-II.docx",
    "ED.A023.MII.&.&&&.01640-II.docx",
    "ED.A023.MII.&.&&&.01641-II.docx",
    "ED.A023.MII.&.&&&.01642-II (eng).docx",
    "ED.A023.MII.&.&&&.01643-II.docx",
    "ED.A023.MII.&.&&&.01644-II (eng).docx",
    "ED.A023.MII.&.&&&.01645-II.docx",
    "ED.A023.MII.&.&&&.01646-II (eng).docx",
    "ED.A023.MII.&.&&&.01648-II (eng).docx",
    "ED.A023.MII.&.&&&.01649-II (eng).docx",
    "ED.A023.MII.&.&&&.01650-II (eng).docx",
    "ED.A023.MII.&.&&&.01651-II.docx",
    "ED.A023.MII.&.&&&.01652-II.docx",
    "ED.A023.MII.&.&&&.01654-II.docx",
    "ED.A023.MII.&.&&&.01655-II.docx",
    "ED.A023.MII.&.&&&.01658-II (eng).docx",
    "ED.A023.MII.&.&&&.01659-II (eng).docx",
    "ED.A023.MII.&.&&&.01660-II.docx",
    "ED.A023.MII.&.&&&.01661-II (eng).docx",
    "ED.A023.MII.&.&&&.01662-II (eng).docx",
    "ED.A023.MII.&.&&&.01663-II (eng).docx",
    "ED.A023.MII.&.&&&.01664-II.docx",
    "ED.A023.MII.&.&&&.01665-II.docx",
    "ED.A023.MII.&.&&&.01666-II.docx",
    "ED.A023.MII.&.&&&.01667-II (eng).docx",
    "ED.A023.MII.&.&&&.01668-II (eng).docx",
    "ED.A023.MII.&.&&&.01669-II (eng).docx",
    "ED.A023.MII.&.&&&.01670-II.docx",
    "ED.A023.MII.&.&&&.01671-II (eng).docx",
    "ED.A023.MII.&.&&&.01672-II (eng).docx",
    "ED.A023.MII.&.&&&.01673-II (eng).docx",
    "ED.A023.MII.&.&&&.01674-II (eng).docx",
    "ED.A023.MII.&.&&&.01675-II (eng).docx",
    "ED.A023.MII.&.&&&.01676-II (eng).docx",
    "ED.A023.MII.&.&&&.01677-II.docx",
    "ED.A023.MII.&.&&&.01678-II (eng).docx",
    "ED.A023.MII.&.&&&.01679-II (eng).docx",
    "ED.A023.MII.&.&&&.01680-II.docx",
    "ED.A023.MII.&.&&&.01681-II (eng).docx",
    "ED.A023.MII.&.&&&.01682-II (eng).docx",
    "ED.A023.MII.&.&&&.01683-II.docx",
    "ED.A023.MII.&.&&&.01684-II.docx",
    "ED.A023.MII.&.&&&.01685-II (eng).docx",
    "ED.A023.MII.&.&&&.01686-II (eng).docx",
    "ED.A023.MII.&.&&&.01687-II.docx",
    "ED.A023.MII.&.&&&.01688-II.docx",
    "ED.A023.MII.&.&&&.01689-II.docx",
    "ED.A023.MII.&.&&&.01690-II (eng).docx",
    "ED.A023.MII.&.&&&.01691-II (eng).docx",
    "ED.A023.MII.&.&&&.01692-II (eng).docx",
    "ED.A023.MII.&.&&&.01693-II.docx",
    "ED.A023.MII.&.&&&.01694-II (eng).docx",
    "ED.A023.MII.&.&&&.01695-II.docx",
    "ED.A023.MII.&.&&&.01696-II.docx",
    "ED.A023.MII.&.&&&.01697-II (eng).docx",
    "ED.A023.MII.&.&&&.01698-II (eng).docx",
    "ED.A023.MII.&.&&&.01702-II (eng).docx",
    "ED.A023.MII.&.&&&.01703-II (eng).docx",
    "ED.A023.MII.&.&&&.01704-II (eng).docx",
    "ED.A023.MII.&.&&&.01705-II.docx",
    "ED.A023.MII.&.&&&.01706-II (eng).docx",
    "ED.A023.MII.&.&&&.01707-II (eng).docx",
    "ED.A023.MII.&.&&&.01708-II (eng).docx",
    "ED.A023.MII.&.&&&.01709-II (eng).docx",
    "ED.A023.MII.&.&&&.01710-II (eng).docx",
    "ED.A023.MII.&.&&&.01711-II.docx",
    "ED.A023.MII.&.&&&.01712-II.docx",
    "ED.A023.MII.&.&&&.01713-II.docx",
    "ED.A023.MII.&.&&&.01714-II (eng).docx",
    "ED.A023.MII.&.&&&.01715-II.docx",
    "ED.A023.MII.&.&&&.01716-II (eng) (изм. даты).docx",
    "ED.A023.MII.&.&&&.01717-II.docx",
    "ED.A023.MII.&.&&&.01718-II (eng).docx",
    "ED.A023.MII.&.&&&.01719-II (eng).docx",
    "ED.A023.MII.&.&&&.01720-II (eng).docx",
    "ED.A023.MII.&.&&&.01721-II (eng).docx",
    "ED.A023.MII.&.&&&.01722-II (eng).docx",
    "ED.A023.MII.&.&&&.01723-II.docx",
    "ED.A023.MII.&.&&&.01724-II.docx",
    "ED.A023.MII.&.&&&.01725-II (eng).docx",
    "ED.A023.MII.&.&&&.01726-II (eng).docx",
    "ED.A023.MII.&.&&&.01727-II.docx",
    "ED.A023.MII.&.&&&.01728-II.docx",
    "ED.A023.MII.&.&&&.01729-II (eng).docx",
    "ED.A023.MII.&.&&&.01730-II.docx",
    "ED.A023.MII.&.&&&.01731-II (eng).docx",
    "ED.A023.MII.&.&&&.01732-II.docx",
    "ED.A023.MII.&.&&&.01733-II.docx",
    "ED.A023.MII.&.&&&.01734-II.docx",
    "ED.A023.MII.&.&&&.01735-II (eng).docx",
    "ED.A023.MII.&.&&&.01736-II.docx",
    "ED.A023.MII.&.&&&.01738-II.docx",
    "ED.A023.MII.&.&&&.01739-II.docx",
    "ED.A023.MII.&.&&&.01740-II.docx",
    "ED.A023.MII.&.&&&.01741-II (eng).docx",
    "ED.A023.MII.&.&&&.01742-II (eng).docx",
    "ED.A023.MII.&.&&&.01743-II (eng).docx",
    "ED.A023.MII.&.&&&.01744-II (eng).docx",
    "ED.A023.MII.&.&&&.01746-II.docx",
    "ED.A023.MII.&.&&&.01747-II.docx",
    "ED.A023.MII.&.&&&.01748-II.docx",
    "ED.A023.MII.&.&&&.01749-II (eng).docx",
    "ED.A023.MII.&.&&&.01750-II (eng).docx",
    "ED.A023.MII.&.&&&.01751-II (eng).docx",
    "ED.A023.MII.&.&&&.01752-II (eng).docx",
    "ED.A023.MII.&.&&&.01753-II (eng).docx",
    "ED.A023.MII.&.&&&.01754-II.docx",
    "ED.A023.MII.&.&&&.01755-II (eng).docx",
    "ED.A023.MII.&.&&&.01756-II (eng).docx",
    "ED.A023.MII.&.&&&.01757-II (eng).docx",
    "ED.A023.MII.&.&&&.01758-II (eng).docx",
    "ED.A023.MII.&.&&&.01763-II.docx",
    "ED.A023.MII.&.&&&.01764-II.docx",
    "ED.A023.MII.&.&&&.01765-II.docx",
    "ED.A023.MII.&.&&&.01766-II.docx",
    "ED.A023.MII.&.&&&.01767-II.docx",
    "ED.A023.MII.&.&&&.01768-II (eng).docx",
    "ED.A023.MII.&.&&&.01769-II (eng).docx",
    "ED.A023.MII.&.&&&.01770-II.docx",
    "ED.A023.MII.&.&&&.01771-II.docx",
    "ED.A023.MII.&.&&&.01772-II.docx",
    "ED.A023.MII.&.&&&.01773-II (eng).docx",
    "ED.A023.MII.&.&&&.01775-II.docx",
    "ED.A023.MII.&.&&&.01776-II.docx",
    "ED.A023.MII.&.&&&.01777-II (eng).docx",
    "ED.A023.MII.&.&&&.01778-II (eng).docx",
    "ED.A023.MII.&.&&&.01779-II.docx",
    "ED.A023.MII.&.&&&.01780-II (eng).docx",
    "ED.A023.MII.&.&&&.01781-II (eng).docx",
    "ED.A023.MII.&.&&&.01782-II (eng).docx",
    "ED.A023.MII.&.&&&.01783-II (eng).docx",
    "ED.A023.MII.&.&&&.01784-II.docx",
    "ED.A023.MII.&.&&&.01785-II (eng).docx",
    "ED.A023.MII.&.&&&.01786-II (eng).docx",
    "ED.A023.MII.&.&&&.01787-II (eng).docx",
    "ED.A023.MII.&.&&&.01788-II.docx",
    "ED.A023.MII.&.&&&.01789-II.docx",
    "ED.A023.MII.&.&&&.01790-II (eng).docx",
    "ED.A023.MII.&.&&&.01791-II (eng).docx",
    "ED.A023.MII.&.&&&.01792-II (eng).docx",
    "ED.A023.MII.&.&&&.01793-II.docx",
    "ED.A023.MII.&.&&&.01794-II.docx",
    "ED.A023.MII.&.&&&.01795-II.docx",
    "ED.A023.MII.&.&&&.01798-II (eng).docx",
    "ED.A023.MII.&.&&&.01799-II (eng).docx",
    "ED.A023.MII.&.&&&.01800-II (eng).docx",
    "ED.A023.MII.&.&&&.01801-II.docx",
    "ED.A023.MII.&.&&&.01802-II.docx",
    "ED.A023.MII.&.&&&.01803-II.docx",
    "ED.A023.MII.&.&&&.01804-II (изм. даты).docx",
    "ED.A023.MII.&.&&&.01805-II (eng).docx",
    "ED.A023.MII.&.&&&.01806-II (eng).docx",
    "ED.A023.MII.&.&&&.01807-II (eng).docx",
    "ED.A023.MII.&.&&&.01808-II.docx",
    "ED.A023.MII.&.&&&.01809-II.docx",
    "ED.A023.MII.&.&&&.01810-II (eng).docx",
    "ED.A023.MII.&.&&&.01810.1-II (eng).docx",
    "ED.A023.MII.&.&&&.01811-II.docx",
    "ED.A023.MII.&.&&&.01812-II.docx",
    "ED.A023.MII.&.&&&.01813-II.docx",
    "ED.A023.MII.&.&&&.01814-II.docx",
    "ED.A023.MII.&.&&&.01816-II.docx",
    "ED.A023.MII.&.&&&.01817-II (eng).docx",
    "ED.A023.MII.&.&&&.01818-II.docx",
    "ED.A023.MII.&.&&&.01819-II.docx",
    "ED.A023.MII.&.&&&.01820-II (eng).docx",
    "ED.A023.MII.&.&&&.01821-II (eng).docx",
    "ED.A023.MII.&.&&&.01823-II (eng).docx",
    "ED.A023.MII.&.&&&.01824-II.docx",
    "ED.A023.MII.&.&&&.01825-II (eng).docx",
    "ED.A023.MII.&.&&&.01826-II (eng).docx",
    "ED.A023.MII.&.&&&.01827-II (eng).docx",
    "ED.A023.MII.&.&&&.01828-II (eng).docx",
    "ED.A023.MII.&.&&&.01829-II (eng).docx",
    "ED.A023.MII.&.&&&.01830-II (eng).docx",
    "ED.A023.MII.&.&&&.01831-II.docx",
    "ED.A023.MII.&.&&&.01832-II.docx",
    "ED.A023.MII.&.&&&.01833-II (eng).docx",
    "ED.A023.MII.&.&&&.01834-II (eng).docx",
    "ED.A023.MII.&.&&&.01835-II.docx",
    "ED.A023.MII.&.&&&.01836-II (eng).docx",
    "ED.A023.MII.&.&&&.01839-II.docx",
    "ED.A023.MII.&.&&&.01841-II.docx",
    "ED.A023.MII.&.&&&.01841-1-II.docx",
    "ED.A023.MII.&.&&&.01842-II.docx",
    "ED.A023.MII.&.&&&.01843-II (eng).docx",
    "ED.A023.MII.&.&&&.01845-II (eng).docx",
    "ED.A023.MII.&.&&&.01847-II (eng).docx",
    "ED.A023.MII.&.&&&.01848-II (eng).docx",
    "ED.A023.MII.&.&&&.01849-II (eng).docx",
    "ED.A023.MII.&.&&&.01850-II (eng).docx",
    "ED.A023.MII.&.&&&.01851-II (eng).docx",
    "ED.A023.MII.&.&&&.01852-II.docx",
    "ED.A023.MII.&.&&&.01853-II (eng).docx",
    "ED.A023.MII.&.&&&.01854-II (eng).docx",
    "ED.A023.MII.&.&&&.01855-II.docx",
    "ED.A023.MII.&.&&&.01856-II.docx",
    "ED.A023.MII.&.&&&.01857-II (eng).docx",
    "ED.A023.MII.&.&&&.01858-II.docx",
    "ED.A023.MII.&.&&&.01859-II (eng).docx",
    "ED.A023.MII.&.&&&.01860-II (eng).docx",
    "ED.A023.MII.&.&&&.01861-II (eng).docx",
    "ED.A023.MII.&.&&&.01863-II (eng).docx",
    "ED.A023.MII.&.&&&.01864-II (eng).docx",
    "ED.A023.MII.&.&&&.01865-II (eng).docx",
    "ED.A023.MII.&.&&&.01866-II.docx",
    "ED.A023.MII.&.&&&.01867-II.docx",
    "ED.A023.MII.&.&&&.01868-II.docx",
    "ED.A023.MII.&.&&&.01869-II.docx",
    "ED.A023.MII.&.&&&.01870-II (eng).docx",
    "ED.A023.MII.&.&&&.01871-II (eng).docx",
    "ED.A023.MII.&.&&&.01872-II (eng).docx",
    "ED.A023.MII.&.&&&.01873-II (eng).docx",
    "ED.A023.MII.&.&&&.01874-II (eng).docx",
    "ED.A023.MII.&.&&&.01875-II.docx",
    "ED.A023.MII.&.&&&.01876-II.docx",
    "ED.A023.MII.&.&&&.01877-II (eng).docx",
    "ED.A023.MII.&.&&&.01878-II (eng).docx",
    "ED.A023.MII.&.&&&.01879-II.docx",
    "ED.A023.MII.&.&&&.01880-II (eng).docx",
    "ED.A023.MII.&.&&&.01881-II.docx",
    "ED.A023.MII.&.&&&.01882-II (eng).docx",
    "ED.A023.MII.&.&&&.01883-II (eng).docx",
    "ED.A023.MII.&.&&&.01884-II (eng).docx",
    "ED.A023.MII.&.&&&.01885-II (eng).docx",
    "ED.A023.MII.&.&&&.01886-II (eng).docx",
    "ED.A023.MII.&.&&&.01887-II (eng).docx",
    "ED.A023.MII.&.&&&.01888-II (eng).docx",
    "ED.A023.MII.&.&&&.01889-II.docx",
    "ED.A023.MII.&.&&&.01890-II.docx",
    "ED.A023.MII.&.&&&.01892-II.docx",
    "ED.A023.MII.&.&&&.01893-II (eng).docx",
    "ED.A023.MII.&.&&&.01896-II.docx",
    "ED.A023.MII.&.&&&.01897-II.docx",
    "ED.A023.MII.&.&&&.01898-II (eng).docx",
    "ED.A023.MII.&.&&&.01899-II (eng).docx",
    "ED.A023.MII.&.&&&.01900-II (eng).docx",
    "ED.A023.MII.&.&&&.01901-II.docx",
    "ED.A023.MII.&.&&&.01902-II.docx",
    "ED.A023.MII.&.&&&.01903-II (eng).docx",
    "ED.A023.MII.&.&&&.01905-II (eng).docx",
    "ED.A023.MII.&.&&&.01906-II.docx",
    "ED.A023.MII.&.&&&.01907-II (eng).docx",
    "ED.A023.MII.&.&&&.01909-II (eng).docx",
    "ED.A023.MII.&.&&&.01910-II (eng).docx",
    "ED.A023.MII.&.&&&.01911-II (eng).docx",
    "ED.A023.MII.&.&&&.01912-II (eng).docx",
    "ED.A023.MII.&.&&&.01913-II (eng).docx",
    "ED.A023.MII.&.&&&.01914-II.docx",
    "ED.A023.MII.&.&&&.01915-II (eng).docx",
    "ED.A023.MII.&.&&&.01916-II (eng).docx",
    "ED.A023.MII.&.&&&.01917-II (eng).docx",
    "ED.A023.MII.&.&&&.01918-II (eng).docx",
    "ED.A023.MII.&.&&&.01919-II (eng).docx",
    "ED.A023.MII.&.&&&.01920-II.docx",
    "ED.A023.MII.&.&&&.01921-II.docx",
    "ED.A023.MII.&.&&&.01922-II.docx",
    "ED.A023.MII.&.&&&.01923-II (eng).docx",
    "ED.A023.MII.&.&&&.01924-II (eng).docx",
    "ED.A023.MII.&.&&&.01925-II (eng).docx",
    "ED.A023.MII.&.&&&.01926-II.docx",
    "ED.A023.MII.&.&&&.01927-II (eng).docx",
    "ED.A023.MII.&.&&&.01928-II.docx",
    "ED.A023.MII.&.&&&.01929-II.docx",
    "ED.A023.MII.&.&&&.01930-II (eng).docx",
    "ED.A023.MII.&.&&&.01931-II (eng).docx",
    "ED.A023.MII.&.&&&.01932-II (eng).docx",
    "ED.A023.MII.&.&&&.01933-II (eng).docx",
    "ED.A023.MII.&.&&&.01934-II (eng).docx",
    "ED.A023.MII.&.&&&.01935-II.docx",
    "ED.A023.MII.&.&&&.01936-II (eng).docx",
    "ED.A023.MII.&.&&&.01937-II (eng).docx",
    "ED.A023.MII.&.&&&.01938-II (eng).docx",
    "ED.A023.MII.&.&&&.01939-II (eng) от 03.06.2026 - в 12.41.docx",
    "ED.A023.MII.&.&&&.01940-II.docx",
    "ED.A023.MII.&.&&&.01941-II.docx",
    "ED.A023.MII.&.&&&.01942-II (eng).docx",
    "ED.A023.MII.&.&&&.01943-II.docx",
    "ED.A023.MII.&.&&&.01944-II (eng).docx",
    "ED.A023.MII.&.&&&.01945-II (eng).docx",
    "ED.A023.MII.&.&&&.01946-II (eng) от 09.06.2026 - в 11.35.docx",
    "ED.A023.MII.&.&&&.01947-II.docx",
    "ED.A023.MII.&.&&&.01948-II.docx",
    "ED.A023.MII.&.&&&.01949-II (eng).docx",
    "ED.A023.MII.&.&&&.01951-II (eng).docx",
    "ED.A023.MII.&.&&&.01952-II.docx",
    "ED.A023.MII.&.&&&.01953-II (eng).docx",
    "ED.A023.MII.&.&&&.01954-II.docx",
    "ED.A023.MII.&.&&&.01955-II (eng).docx",
    "ED.A023.MII.&.&&&.01956-II (eng).docx",
    "ED.A023.MII.&.&&&.01957-II.docx",
    "ED.A023.MII.&.&&&.01958-II (eng).docx",
    "ED.A023.MII.&.&&&.01959-II (eng).docx",
    "ED.A023.MII.&.&&&.01960-II (eng).docx",
    "ED.A023.MII.&.&&&.01961-II.docx",
    "ED.A023.MII.&.&&&.01962-II (eng).docx",
    "ED.A023.MII.&.&&&.01963-II (eng).docx",
    "ED.A023.MII.&.&&&.01964-II.docx",
    "ED.A023.MII.&.&&&.01965-II (eng).docx",
    "ED.A023.MII.&.&&&.01966-II.docx",
    "ED.A023.MII.&.&&&.01967-II (eng).docx",
    "ED.A023.MII.&.&&&.01968-II (eng).docx",
    "ED.A023.MII.&.&&&.01970-II.docx",
    "ED.A023.MII.&.&&&.01971-II (eng).docx",
    "ED.A023.MII.&.&&&.01972-II (eng).docx",
    "ED.A023.MII.&.&&&.01973-II (eng).docx",
    "ED.A023.MII.&.&&&.01974-II.docx",
    "ED.A023.MII.&.&&&.01975-II.docx",
    "ED.A023.MII.&.&&&.01976-II.docx",
    "ED.A023.MII.&.&&&.01977-II (eng).docx",
    "ED.A023.MII.&.&&&.01978-II (eng).docx",
    "ED.A023.MII.&.&&&.01979-II (eng).docx",
    "ED.A023.MII.&.&&&.01980-II.docx",
    "ED.A023.MII.&.&&&.01981-II.docx",
    "ED.A023.MII.&.&&&.01982-II (eng).docx",
    "ED.A023.MII.&.&&&.01985-II (eng).docx",
    "ED.A023.MII.&.&&&.01986-II (eng).docx",
    "ED.A023.MII.&.&&&.01987-II (eng).docx",
    "ED.A023.MII.&.&&&.01988-II.docx",
    "ED.A023.MII.&.&&&.01989-II.docx",
    "ED.A023.MII.&.&&&.01990-II (eng).docx",
    "ED.A023.MII.&.&&&.01991-II (eng).docx",
    "ED.A023.MII.&.&&&.01992-II (eng).docx",
    "ED.A023.MII.&.&&&.01993-II (eng).docx",
    "ED.A023.MII.&.&&&.01994-II (eng).docx",
    "ED.A023.MII.&.&&&.01995-II (eng).docx",
    "ED.A023.MII.&.&&&.01996-II (eng).docx",
    "ED.A023.MII.&.&&&.01997-II.docx",
    "ED.A023.MII.&.&&&.01998-II.docx",
    "ED.A023.MII.&.&&&.01999-II.docx",
    "ED.A023.MII.&.&&&.02000-II (eng).docx",
    "ED.A023.MII.&.&&&.02001-II.docx",
    "ED.A023.MII.&.&&&.02002-II (eng).docx",
    "ED.A023.MII.&.&&&.02003-II.docx",
    "ED.A023.MII.&.&&&.02004-II (eng).docx",
    "ED.A023.MII.&.&&&.02005-II (eng).docx",
    "ED.A023.MII.&.&&&.02006-II.docx",
    "ED.A023.MII.&.&&&.02007-II.docx",
    "ED.A023.MII.&.&&&.02008-II.docx",
    "ED.A023.MII.&.&&&.02009-II (eng).docx",
    "ED.A023.MII.&.&&&.02010-1-II.docx",
    "ED.A023.MII.&.&&&.02010-II.docx",
    "ED.A023.MII.&.&&&.02011-II (eng).docx",
    "ED.A023.MII.&.&&&.02012-II (eng).docx",
    "ED.A023.MII.&.&&&.02014-II (eng).docx",
    "ED.A023.MII.&.&&&.02015-II (eng).docx",
    "ED.A023.MII.&.&&&.02016-II (eng).docx",
    "ED.A023.MII.&.&&&.02017-II (eng).docx",
    "ED.A023.MII.&.&&&.02018-II (eng).docx",
    "ED.A023.MII.&.&&&.02019-II.docx",
    "ED.A023.MII.&.&&&.02020-II.docx",
    "ED.A023.MII.&.&&&.02021-II (eng).docx",
    "ED.A023.MII.&.&&&.02022-II (eng).docx",
    "ED.A023.MII.&.&&&.02024-II (eng).docx",
    "ED.A023.MII.&.&&&.02025-II (eng).docx",
    "ED.A023.MII.&.&&&.02026-II (eng).docx",
    "ED.A023.MII.&.&&&.02027-II (eng).docx",
    "ED.A023.MII.&.&&&.02028-II (eng).docx",
    "ED.A023.MII.&.&&&.02029-II (eng).docx",
    "ED.A023.MII.&.&&&.02030-II (eng).docx",
    "ED.A023.MII.&.&&&.02031-II.docx",
    "ED.A023.MII.&.&&&.02032-II.docx",
    "ED.A023.MII.&.&&&.02033-II (eng).docx",
    "ED.A023.MII.&.&&&.02034-II.docx",
    "ED.A023.MII.&.&&&.02035-II (eng).docx",
    "ED.A023.MII.&.&&&.02036-II (eng).docx",
    "ED.A023.MII.&.&&&.02037-II (eng).docx",
    "ED.A023.MII.&.&&&.02038-II.docx",
    "ED.A023.MII.&.&&&.02039-II.docx",
    "ED.A023.MII.&.&&&.02040-II (eng).docx",
    "ED.A023.MII.&.&&&.02041-II.docx",
    "ED.A023.MII.&.&&&.02042-II (eng).docx",
    "ED.A023.MII.&.&&&.02045-II.docx",
    "ED.A023.MII.&.&&&.02046-II.docx",
    "ED.A023.MII.&.&&&.02047-II.docx",
    "ED.A023.MII.&.&&&.02048-II (eng).docx",
    "ED.A023.MII.&.&&&.02049-II (eng).docx",
    "ED.A023.MII.&.&&&.02050-II.docx",
    "ED.A023.MII.&.&&&.02051-II (eng).docx",
    "ED.A023.MII.&.&&&.02053-II (eng).docx",
    "ED.A023.MII.&.&&&.02054-II.docx",
    "ED.A023.MII.&.&&&.02055-II.docx",
    "ED.A023.MII.&.&&&.02056-II.docx",
    "ED.A023.MII.&.&&&.02057-II (eng).docx",
    "ED.A023.MII.&.&&&.02058-II (eng).docx",
    "ED.A023.MII.&.&&&.02059-II (eng).docx",
    "ED.A023.MII.&.&&&.02060-II (eng).docx",
    "ED.A023.MII.&.&&&.02061-II.docx",
    "ED.A023.MII.&.&&&.02062-II (eng).docx",
    "ED.A023.MII.&.&&&.02063-II (eng).docx",
    "ED.A023.MII.&.&&&.02064-II (eng).docx",
    "ED.A023.MII.&.&&&.02065-II.docx",
    "ED.A023.MII.&.&&&.02066-II (eng).docx",
    "ED.A023.MII.&.&&&.02067-II (eng).docx",
    "ED.A023.MII.&.&&&.02068-II.docx",
    "ED.A023.MII.&.&&&.02069-II (eng).docx",
    "ED.A023.MII.&.&&&.02070-II (eng).docx",
    "ED.A023.MII.&.&&&.02071-II (eng).docx",
    "ED.A023.MII.&.&&&.02072-II (eng).docx",
    "ED.A023.MII.&.&&&.02073-II (eng).docx",
    "ED.A023.MII.&.&&&.02074-II.docx",
    "ED.A023.MII.&.&&&.02075-II (eng).docx",
    "ED.A023.MII.&.&&&.02076-II (eng).docx",
    "ED.A023.MII.&.&&&.02077-II (eng).docx",
    "ED.A023.MII.&.&&&.02078-II (eng).docx",
    "ED.A023.MII.&.&&&.02079-II (eng).docx",
    "ED.A023.MII.&.&&&.02082-II (eng).docx",
    "ED.A023.MII.&.&&&.02083-II.docx",
    "ED.A023.MII.&.&&&.02084-II (eng).docx",
    "ED.A023.MII.&.&&&.02085-II (eng).docx",
    "ED.A023.MII.&.&&&.02086-II (eng).docx",
    "ED.A023.MII.&.&&&.02087-II.docx",
    "ED.A023.MII.&.&&&.02088-II (eng).docx",
    "ED.A023.MII.&.&&&.02089-II (eng).docx",
    "ED.A023.MII.&.&&&.02090-II (eng).docx",
    "ED.A023.MII.&.&&&.02091-II (eng).docx",
    "ED.A023.MII.&.&&&.02092-II (eng).docx",
    "ED.A023.MII.&.&&&.02093-II.docx",
    "ED.A023.MII.&.&&&.02094-II.docx",
    "ED.A023.MII.&.&&&.02095-II.docx",
    "ED.A023.MII.&.&&&.02096-II (eng).docx",
    "ED.A023.MII.&.&&&.02097-II (eng).docx",
    "ED.A023.MII.&.&&&.02098-II (eng).docx",
    "ED.A023.MII.&.&&&.02099-II.docx",
    "ED.A023.MII.&.&&&.02100-II.docx",
    "ED.A023.MII.&.&&&.02101-II.docx",
    "ED.A023.MII.&.&&&.02102-II (eng).docx",
    "ED.A023.MII.&.&&&.02103-II (eng).docx",
    "ED.A023.MII.&.&&&.02104-II (eng).docx",
    "ED.A023.MII.&.&&&.02105-II (eng).docx",
    "ED.A023.MII.&.&&&.02106-II.docx",
    "ED.A023.MII.&.&&&.02107-II (eng).docx",
    "ED.A023.MII.&.&&&.02108-II (eng).docx",
    "ED.A023.MII.&.&&&.02109-II.docx",
    "ED.A023.MII.&.&&&.02110-II (eng).docx",
    "ED.A023.MII.&.&&&.02111-II.docx",
    "ED.A023.MII.&.&&&.02115-II (eng).docx",
    "ED.A023.MII.&.&&&.02116-II (eng).docx",
    "ED.A023.MII.&.&&&.02117-II (eng).docx",
    "ED.A023.MII.&.&&&.02118-II (eng).docx",
    "ED.A023.MII.&.&&&.02119-II (eng).docx",
    "ED.A023.MII.&.&&&.02120-II (eng).docx",
    "ED.A023.MII.&.&&&.02121-II (eng).docx",
    "ED.A023.MII.&.&&&.02122-II (eng).docx",
    "ED.A023.MII.&.&&&.02123-II (eng).docx",
    "ED.A023.MII.&.&&&.02124-II.docx",
    "ED.A023.MII.&.&&&.02125-II (eng).docx",
    "ED.A023.MII.&.&&&.02126-II (eng).docx",
    "ED.A023.MII.&.&&&.02127-II (eng).docx",
    "ED.A023.MII.&.&&&.02128-II.docx",
    "ED.A023.MII.&.&&&.02130-II.docx",
    "ED.A023.MII.&.&&&.02131-II (eng).docx",
    "ED.A023.MII.&.&&&.02132-II (eng).docx",
    "ED.A023.MII.&.&&&.02133-II (eng).docx",
    "ED.A023.MII.&.&&&.02134-II (eng).docx",
    "ED.A023.MII.&.&&&.02135-II (eng).docx",
    "ED.A023.MII.&.&&&.02136-II.docx",
    "ED.A023.MII.&.&&&.02137-II.docx",
    "ED.A023.MII.&.&&&.02138-II (eng).docx",
    "ED.A023.MII.&.&&&.02139-II.docx",
    "ED.A023.MII.&.&&&.02140-II.docx",
    "ED.A023.MII.&.&&&.02141-II (eng).docx",
    "ED.A023.MII.&.&&&.02143-II (eng).docx",
    "ED.A023.MII.&.&&&.02144-II (eng).docx",
    "ED.A023.MII.&.&&&.02145-II (eng) от 04.10.2026 - в 16.31.docx",
    "ED.A023.MII.&.&&&.02146-II (eng).docx",
    "ED.A023.MII.&.&&&.02147-II (eng).docx",
    "ED.A023.MII.&.&&&.02148-II (eng).docx",
    "ED.A023.MII.&.&&&.02149-II (eng).docx",
    "ED.A023.MII.&.&&&.02150-II (eng).docx",
    "ED.A023.MII.&.&&&.02151-II (eng).docx",
    "ED.A023.MII.&.&&&.02152-II.docx",
    "ED.A023.MII.&.&&&.02153-II.docx",
    "ED.A023.MII.&.&&&.02154-II (eng).docx",
    "ED.A023.MII.&.&&&.02155-II (eng).docx",
    "ED.A023.MII.&.&&&.02156-II (eng) от 05.10.2026 - в 09.15.docx",
    "ED.A023.MII.&.&&&.02157-II (eng).docx",
    "ED.A023.MII.&.&&&.02158-II (eng).docx",
    "ED.A023.MII.&.&&&.02159-II (eng).docx",
    "ED.A023.MII.&.&&&.02160-II.docx",
    "ED.A023.MII.&.&&&.02161-II (eng).docx",
    "ED.A023.MII.&.&&&.02162-II (eng).docx",
    "ED.A023.MII.&.&&&.02163-II (eng).docx",
    "ED.A023.MII.&.&&&.02164-II (eng).docx",
    "ED.A023.MII.&.&&&.02165-II.docx",
    "ED.A023.MII.&.&&&.02166-II.docx",
    "ED.A023.MII.&.&&&.02167-II (eng).docx",
    "ED.A023.MII.&.&&&.02168-II (eng).docx",
    "ED.A023.MII.&.&&&.02169-II (eng).docx",
    "ED.A023.MII.&.&&&.02170-II (eng).docx",
    "ED.A023.MII.&.&&&.02171-II (eng).docx",
    "ED.A023.MII.&.&&&.02172-II.docx",
    "ED.A023.MII.&.&&&.02173-II (eng).docx",
    "ED.A023.MII.&.&&&.02174-II (eng).docx",
    "ED.A023.MII.&.&&&.02175-II (eng).docx",
    "ED.A023.MII.&.&&&.02176-II (eng).docx",
    "ED.A023.MII.&.&&&.02177-II.docx",
    "ED.A023.MII.&.&&&.02178-II (eng).docx",
    "ED.A023.MII.&.&&&.02180-II.docx",
    "ED.A023.MII.&.&&&.02181-II (eng).docx",
    "ED.A023.MII.&.&&&.02182-II.docx",
    "ED.A023.MII.&.&&&.02183-II (eng).docx",
    "ED.A023.MII.&.0009 (eng).docx",
    "ED.A023.MII.&.012.01528-II (eng).docx",
    "ED.A023.MII.&.012.01579-II (eng).docx",
    "ED.A023.MII.&.012.01589-II (eng).docx",
    "ED.A023.MII.&.012.01604-II (eng).docx",
    "ED.A023.MII.&.012.01605-II (eng).docx",
    "ED.A023.MII.&.012.01606-II (eng).docx",
    "ED.A023.MII.&.012.01607-II (eng).docx",
    "ED.A023.MII.&.012.01699-II (eng).docx",
    "ED.A023.MII.&.012.01745-II (eng).docx",
    "ED.A023.MII.&.012.01759-II (eng).docx",
    "ED.A023.MII.&.012.01760-II (eng).docx",
    "ED.A023.MII.&.012.01761-II (eng).docx",
    "ED.A023.MII.&.012.01762-1-II (eng).docx",
    "ED.A023.MII.&.012.01762-2-II (eng).docx",
    "ED.A023.MII.&.012.01774-II (eng).docx",
    "ED.A023.MII.&.012.01837-II (eng).docx",
    "ED.A023.MII.&.012.01844-II (eng).docx",
    "ED.A023.MII.&.012.01862-II (eng).docx",
    "ED.A023.MII.&.012.01894-II (eng).docx",
    "ED.A023.MII.&.012.01895-II (eng).docx",
    "ED.A023.MII.&.012.01983-II (eng).docx",
    "ED.A023.MII.&.012.01984-II (eng).docx",
    "ED.A023.MII.&.012.02043-II (eng).docx",
    "ED.A023.MII.&.012.02044-II (eng).docx",
    "ED.A023.MII.&.012.02080-II (eng).docx",
    "ED.A023.MII.&.012.02081-II (eng).docx",
    "ED.A023.MII.&.012.02112-II (eng).docx",
    "ED.A023.MII.&.012.02113-II (eng).docx",
    "ED.A023.MII.&.012.02114-II (eng) от 08.09.2026 - в 08.47.docx",
    "ED.A023.MII.&.012.02129-II (eng).docx",
    "ED.A023.MII.&.012.02142-II (eng).docx",
    "ED.A023.MII.1.0002-1 (eng).docx",
    "ED.A023.MII.1.012.0910-1 (eng) от 21.06.2024 - в 11.51.docx",
    "ED.A023.MII.2.012.0910-2 (eng).docx"
]